# Name That LLM: CPU baseline

Self-contained project export. Attach the official competition data,
select **Internet off**, and **Save & Run All**. This fits the local
TF-IDF/style baseline and writes `submission.parquet`. No API calls or
model downloads run in this notebook. A successful sample run checks the
pipeline; only an official submission measures hidden-test accuracy.


In [ ]:
# Embedded project source; regenerate this notebook after editing src/ntl.
import hashlib
import importlib
import json
import os
from pathlib import Path
import sys

# Scoring is offline. These also prevent accidental Hugging Face requests.
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

SOURCE_JSON = '{"ntl/__init__.py": "\\"\\"\\"Reproducible, offline-friendly tools for the Name That LLM competition.\\"\\"\\"\\n\\n__version__ = \\"0.1.0\\"\\n", "ntl/__main__.py": "from .cli import main\\n\\nraise SystemExit(main())\\n", "ntl/cli.py": "\\"\\"\\"Command-line entry points; no credentials, network calls or submissions.\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport argparse\\nimport importlib.metadata\\nimport json\\nimport platform\\nimport sys\\nfrom pathlib import Path\\n\\nfrom .constants import DEFAULT_SEED\\n\\n\\ndef _model_arguments(parser: argparse.ArgumentParser) -> None:\\n    parser.add_argument(\\"--train\\", required=True, type=Path)\\n    parser.add_argument(\\"--output-dir\\", required=True, type=Path)\\n    parser.add_argument(\\"--seed\\", type=int, default=DEFAULT_SEED)\\n    parser.add_argument(\\"--char-max-features\\", type=int, default=50000)\\n    parser.add_argument(\\"--word-max-features\\", type=int, default=30000)\\n    parser.add_argument(\\"--C\\", type=float, default=4.0)\\n    parser.add_argument(\\"--max-iter\\", type=int, default=1000)\\n\\n\\ndef parser() -> argparse.ArgumentParser:\\n    result = argparse.ArgumentParser(description=\\"Name That LLM: offline data checks and CPU baseline\\")\\n    subcommands = result.add_subparsers(dest=\\"command\\", required=True)\\n    subcommands.add_parser(\\"doctor\\", help=\\"Report interpreter and dependency versions\\")\\n    check = subcommands.add_parser(\\"data-check\\", help=\\"Validate official data schemas without altering text\\")\\n    check.add_argument(\\"--train\\", required=True, type=Path)\\n    check.add_argument(\\"--test\\", type=Path)\\n    cv = subcommands.add_parser(\\"baseline-cv\\", help=\\"Evaluate with prompt-grouped folds and save OOF predictions\\")\\n    _model_arguments(cv)\\n    cv.add_argument(\\"--folds\\", type=int, default=5)\\n    cv.add_argument(\\"--fold-manifest\\", type=Path)\\n    training = subcommands.add_parser(\\"baseline-train\\", help=\\"Fit on all labeled data and save an inference artifact\\")\\n    _model_arguments(training)\\n    predict = subcommands.add_parser(\\"predict\\", help=\\"Write a validated submission.parquet from a trusted local model\\")\\n    predict.add_argument(\\"--model\\", required=True, type=Path)\\n    predict.add_argument(\\"--test\\", required=True, type=Path)\\n    predict.add_argument(\\"--output\\", required=True, type=Path)\\n    smoke = subcommands.add_parser(\\"smoke\\", help=\\"Exercise the pipeline on clearly synthetic fixtures\\")\\n    smoke.add_argument(\\"--output-dir\\", required=True, type=Path)\\n    return result\\n\\n\\ndef main(argv: list[str] | None = None) -> int:\\n    args = parser().parse_args(argv)\\n    try:\\n        if args.command == \\"doctor\\":\\n            versions = {}\\n            for package in (\\"pandas\\", \\"numpy\\", \\"scipy\\", \\"scikit-learn\\", \\"joblib\\", \\"pyarrow\\"):\\n                try:\\n                    versions[package] = importlib.metadata.version(package)\\n                except importlib.metadata.PackageNotFoundError:\\n                    versions[package] = None\\n            print(json.dumps({\\"python\\": platform.python_version(), \\"executable\\": sys.executable,\\n                              \\"dependencies\\": versions}, indent=2))\\n            return int(any(value is None for value in versions.values()))\\n        if args.command == \\"data-check\\":\\n            from .data import data_quality_report, fingerprint_frame, read_test, read_train\\n            train = read_train(args.train)\\n            summary = {\\"training_rows\\": len(train), \\"prompts\\": int(train[\\"prompt_id\\"].nunique()),\\n                       \\"class_counts\\": train[\\"model\\"].value_counts().sort_index().to_dict(),\\n                       \\"train_fingerprint\\": fingerprint_frame(train),\\n                       \\"data_quality\\": data_quality_report(train)}\\n            if args.test:\\n                test = read_test(args.test)\\n                if set(train[\\"id\\"]) & set(test[\\"id\\"]):\\n                    raise ValueError(\\"Train and test passage ids overlap\\")\\n                summary[\\"test_rows\\"] = len(test)\\n            print(json.dumps(summary, indent=2))\\n            return 0\\n        if args.command == \\"smoke\\":\\n            from .smoke import run_smoke\\n            summary = run_smoke(args.output_dir)\\n        elif args.command in {\\"baseline-cv\\", \\"baseline-train\\"}:\\n            from .data import read_train\\n            from .modeling import BaselineConfig, run_cross_validation, train_baseline\\n            config = BaselineConfig(seed=args.seed, char_max_features=args.char_max_features,\\n                                    word_max_features=args.word_max_features, C=args.C, max_iter=args.max_iter)\\n            train = read_train(args.train)\\n            if args.command == \\"baseline-cv\\":\\n                summary = run_cross_validation(train, args.output_dir, n_splits=args.folds,\\n                                               config=config, fold_manifest=args.fold_manifest)\\n            else:\\n                summary = {\\"model_path\\": str(train_baseline(train, args.output_dir, config=config))}\\n        else:\\n            from .data import read_test\\n            from .modeling import predict_baseline\\n            submission = predict_baseline(args.model, read_test(args.test), args.output)\\n            summary = {\\"submission_path\\": str(args.output), \\"rows\\": len(submission)}\\n        print(json.dumps(summary, indent=2))\\n        return 0\\n    except (ValueError, OSError, ImportError) as error:\\n        print(f\\"Error: {error}\\", file=sys.stderr)\\n        return 2\\n", "ntl/constants.py": "\\"\\"\\"Competition labels, in an explicit order shared by every model artifact.\\"\\"\\"\\n\\nLABELS = (\\n    \\"gpt-5.5\\",\\n    \\"gpt-5.6-luna\\",\\n    \\"gpt-5.6-sol\\",\\n    \\"gpt-5.6-terra\\",\\n    \\"gpt-6-astra\\",\\n    \\"gemini-3.1-pro-preview\\",\\n    \\"gemini-3.6-flash\\",\\n    \\"gemini-3.7-flash\\",\\n    \\"gemini-3.8-flash\\",\\n    \\"claude-fable-5\\",\\n    \\"claude-opus-4-6\\",\\n    \\"claude-opus-4-8\\",\\n    \\"claude-opus-5\\",\\n    \\"claude-sonnet-5\\",\\n    \\"qwen3.8-max\\",\\n    \\"glm-5.3\\",\\n    \\"glm-5.3-flash\\",\\n    \\"kimi-k3\\",\\n)\\n\\nDEFAULT_SEED = 2026\\n", "ntl/data.py": "\\"\\"\\"Strict schemas and content fingerprints. Text is never normalized here.\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport hashlib\\nimport json\\nfrom pathlib import Path\\nfrom typing import Sequence\\n\\nimport pandas as pd\\n\\nfrom .constants import LABELS\\n\\nTRAIN_COLUMNS = (\\"id\\", \\"prompt_id\\", \\"text\\", \\"model\\")\\nTEST_COLUMNS = (\\"id\\", \\"text\\")\\n\\n\\ndef _strings(frame: pd.DataFrame, columns: Sequence[str]) -> None:\\n    missing = set(columns) - set(frame.columns)\\n    if missing:\\n        raise ValueError(f\\"Missing columns: {sorted(missing)}\\")\\n    if frame.columns.duplicated().any():\\n        raise ValueError(\\"Duplicate column names are not allowed\\")\\n    for column in columns:\\n        values = frame[column]\\n        if values.isna().any():\\n            raise ValueError(f\\"{column} contains null values\\")\\n        if not values.map(lambda value: isinstance(value, str)).all():\\n            raise ValueError(f\\"{column} must contain strings\\")\\n        if values.map(lambda value: not value.strip()).any():\\n            raise ValueError(f\\"{column} contains empty or whitespace-only strings\\")\\n    if frame[\\"id\\"].duplicated().any():\\n        raise ValueError(\\"Duplicate passage ids are not allowed\\")\\n\\n\\ndef validate_train(frame: pd.DataFrame, require_all_labels: bool = True) -> pd.DataFrame:\\n    \\"\\"\\"Validate labeled training rows without changing their text or order.\\"\\"\\"\\n    _strings(frame, TRAIN_COLUMNS)\\n    if frame.empty:\\n        raise ValueError(\\"Training data is empty\\")\\n    observed = set(frame[\\"model\\"])\\n    unknown = observed - set(LABELS)\\n    if unknown:\\n        raise ValueError(f\\"Unknown model labels: {sorted(unknown)}\\")\\n    if require_all_labels and observed != set(LABELS):\\n        raise ValueError(f\\"Training data is missing model labels: {sorted(set(LABELS) - observed)}\\")\\n    return frame.copy()\\n\\n\\ndef validate_test(frame: pd.DataFrame) -> pd.DataFrame:\\n    \\"\\"\\"Validate the hidden-test schema and reject accidental target leakage.\\"\\"\\"\\n    _strings(frame, TEST_COLUMNS)\\n    if frame.empty:\\n        raise ValueError(\\"Test data is empty; expected at least one passage\\")\\n    forbidden = {\\"model\\", \\"prompt_id\\"} & set(frame.columns)\\n    if forbidden:\\n        raise ValueError(f\\"Test data must not contain training-only columns: {sorted(forbidden)}\\")\\n    return frame.copy()\\n\\n\\ndef data_quality_report(train: pd.DataFrame) -> dict:\\n    \\"\\"\\"Describe identical outputs without rejecting legitimate shared responses.\\n\\n    Text shared between prompts can make prompt-grouped validation optimistic;\\n    cross-label identical outputs can also impose an attribution ambiguity.\\n    Neither proves a schema error, so report both rather than removing rows.\\n    \\"\\"\\"\\n    train = validate_train(train)\\n    grouped = train.groupby(\\"text\\", sort=False).agg(\\n        rows=(\\"id\\", \\"size\\"), labels=(\\"model\\", \\"nunique\\"), prompts=(\\"prompt_id\\", \\"nunique\\")\\n    )\\n    repeated = grouped[\\"rows\\"] > 1\\n    ambiguous = grouped[\\"labels\\"] > 1\\n    cross_prompt = grouped[\\"prompts\\"] > 1\\n    warnings = []\\n    if cross_prompt.any():\\n        warnings.append(\\n            \\"Identical text appears under multiple prompt_ids; prompt-grouped validation may be optimistic.\\"\\n        )\\n    if ambiguous.any():\\n        warnings.append(\\"Identical text has multiple model labels; these rows may be intrinsically ambiguous.\\")\\n    return {\\n        \\"unique_texts\\": len(grouped),\\n        \\"exact_duplicate_text_groups\\": int(repeated.sum()),\\n        \\"exact_duplicate_text_rows\\": int(grouped.loc[repeated, \\"rows\\"].sum()),\\n        \\"exact_duplicate_text_excess_rows\\": int((grouped.loc[repeated, \\"rows\\"] - 1).sum()),\\n        \\"cross_label_identical_text_groups\\": int(ambiguous.sum()),\\n        \\"cross_label_identical_text_rows\\": int(grouped.loc[ambiguous, \\"rows\\"].sum()),\\n        \\"cross_prompt_identical_text_groups\\": int(cross_prompt.sum()),\\n        \\"cross_prompt_identical_text_rows\\": int(grouped.loc[cross_prompt, \\"rows\\"].sum()),\\n        \\"warnings\\": warnings,\\n    }\\n\\n\\ndef _read_csv(path: str | Path) -> pd.DataFrame:\\n    # keep_default_na=False preserves legitimate text such as \\"NA\\" or \\"null\\".\\n    # Empty cells are rejected by the schema, rather than silently imputed.\\n    return pd.read_csv(path, dtype=str, keep_default_na=False)\\n\\n\\ndef read_train(path: str | Path, require_all_labels: bool = True) -> pd.DataFrame:\\n    return validate_train(_read_csv(path), require_all_labels=require_all_labels)\\n\\n\\ndef read_test(path: str | Path) -> pd.DataFrame:\\n    return validate_test(_read_csv(path))\\n\\n\\ndef fingerprint_frame(frame: pd.DataFrame) -> str:\\n    \\"\\"\\"SHA-256 of exact required training content, invariant to CSV row order.\\"\\"\\"\\n    validated = validate_train(frame, require_all_labels=False)\\n    digest = hashlib.sha256()\\n    digest.update(json.dumps(TRAIN_COLUMNS).encode(\\"utf-8\\"))\\n    ordered = validated.sort_values(\\"id\\", kind=\\"stable\\")\\n    for row in ordered.loc[:, list(TRAIN_COLUMNS)].itertuples(index=False, name=None):\\n        digest.update(b\\"\\\\n\\")\\n        digest.update(json.dumps(row, ensure_ascii=False, separators=(\\",\\", \\":\\")).encode(\\"utf-8\\"))\\n    return digest.hexdigest()\\n\\n\\ndef validate_submission(\\n    submission: pd.DataFrame,\\n    expected_ids: Sequence[str] | pd.Series | pd.DataFrame,\\n) -> pd.DataFrame:\\n    \\"\\"\\"Check columns, labels and exact id coverage, independent of row order.\\"\\"\\"\\n    if set(submission.columns) != {\\"id\\", \\"model\\"}:\\n        raise ValueError(\\"Submission must contain exactly the id and model columns\\")\\n    _strings(submission, (\\"id\\", \\"model\\"))\\n    if submission.empty:\\n        raise ValueError(\\"Submission is empty\\")\\n    if isinstance(expected_ids, pd.DataFrame):\\n        expected_ids = expected_ids[\\"id\\"]\\n    expected = list(expected_ids)\\n    if any(not isinstance(value, str) or not value.strip() for value in expected):\\n        raise ValueError(\\"Expected ids must be nonempty strings\\")\\n    if len(set(expected)) != len(expected):\\n        raise ValueError(\\"Expected ids contain duplicates\\")\\n    if len(submission) != len(expected) or set(submission[\\"id\\"]) != set(expected):\\n        raise ValueError(\\"Submission ids do not exactly match the test ids\\")\\n    unknown = set(submission[\\"model\\"]) - set(LABELS)\\n    if unknown:\\n        raise ValueError(f\\"Submission contains unknown labels: {sorted(unknown)}\\")\\n    return submission.copy()\\n\\n\\ndef write_submission(\\n    ids: Sequence[str] | pd.Series,\\n    predictions: Sequence[str],\\n    path: str | Path,\\n) -> pd.DataFrame:\\n    ids = list(ids)\\n    predictions = list(predictions)\\n    if len(ids) != len(predictions):\\n        raise ValueError(\\"Prediction count does not match test row count\\")\\n    submission = pd.DataFrame({\\"id\\": ids, \\"model\\": predictions})\\n    validate_submission(submission, ids)\\n    destination = Path(path)\\n    if destination.suffix != \\".parquet\\":\\n        raise ValueError(\\"Competition submissions must be Parquet files\\")\\n    destination.parent.mkdir(parents=True, exist_ok=True)\\n    submission.to_parquet(destination, index=False)\\n    validate_submission(pd.read_parquet(destination), ids)\\n    return submission\\n", "ntl/encoders.py": "\\"\\"\\"Frozen, pinned StyleDistance and Qwen3 embeddings for the Gemini diagnostic.\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport argparse\\nimport gc\\nimport importlib.metadata\\nimport json\\nimport time\\nfrom pathlib import Path\\n\\nimport numpy as np\\nimport pandas as pd\\n\\nfrom .data import read_train\\nfrom .frozen_gemini import save_feature_archive\\nfrom .gemini import (\\n    OFFICIAL_BASELINE_COUNTS,\\n    _initialize_cuda_memory_stats,\\n    prepare_gemini_evaluation,\\n)\\n\\nENCODERS = {\\n    \\"styledistance\\": {\\n        \\"model_id\\": \\"StyleDistance/styledistance\\",\\n        \\"model_revision\\": \\"b7df5f0b0480773c097ba3121d83ca32b71015ca\\",\\n        \\"model_type\\": \\"roberta\\", \\"layers\\": 12, \\"dimension\\": 768,\\n        \\"pooling\\": \\"masked_mean_including_special_tokens\\", \\"padding_side\\": \\"right\\",\\n        \\"st_modules\\": [\\"Transformer\\", \\"Pooling\\"],\\n    },\\n    \\"qwen3-embedding-0.6b\\": {\\n        \\"model_id\\": \\"Qwen/Qwen3-Embedding-0.6B\\",\\n        \\"model_revision\\": \\"97b0c614be4d77ee51c0cef4e5f07c00f9eb65b3\\",\\n        \\"model_type\\": \\"qwen3\\", \\"layers\\": 28, \\"dimension\\": 1024,\\n        \\"pooling\\": \\"last_non_padding_token\\", \\"padding_side\\": \\"left\\",\\n        \\"st_modules\\": [\\"Transformer\\", \\"Pooling\\", \\"Normalize\\"],\\n    },\\n}\\n\\n\\ndef validate_snapshot(directory: str | Path, model_key: str) -> dict:\\n    spec = ENCODERS[model_key]\\n    directory = Path(directory)\\n    manifest = json.loads((directory / \\"snapshot_manifest.json\\").read_text())\\n    if (manifest.get(\\"schema_version\\") != 1\\n            or manifest.get(\\"repo_id\\") != spec[\\"model_id\\"]\\n            or manifest.get(\\"revision\\") != spec[\\"model_revision\\"]\\n            or manifest.get(\\"includes_weights\\") is not True):\\n        raise ValueError(\\"Expected a full snapshot of the selected immutable encoder revision\\")\\n    entries = manifest.get(\\"files\\", [])\\n    names = set()\\n    if not entries:\\n        raise ValueError(\\"Encoder snapshot has no recorded files\\")\\n    for entry in entries:\\n        relative = Path(entry[\\"path\\"])\\n        if relative.is_absolute() or \\"..\\" in relative.parts or str(relative) in names:\\n            raise ValueError(\\"Unsafe or repeated encoder snapshot path\\")\\n        names.add(str(relative))\\n        path = directory / relative\\n        if path.is_symlink() or not path.is_file() or path.stat().st_size != entry[\\"size\\"]:\\n            raise ValueError(f\\"Missing or changed encoder snapshot file: {relative}\\")\\n    if not any(name.endswith(\\".safetensors\\") for name in names):\\n        raise ValueError(\\"The pinned encoders require safetensors weights\\")\\n    config = json.loads((directory / \\"config.json\\").read_text())\\n    actual = (config.get(\\"model_type\\"), config.get(\\"num_hidden_layers\\"), config.get(\\"hidden_size\\"))\\n    if actual != (spec[\\"model_type\\"], spec[\\"layers\\"], spec[\\"dimension\\"]):\\n        raise ValueError(\\"Encoder architecture differs from the reviewed checkpoint\\")\\n    modules = json.loads((directory / \\"modules.json\\").read_text())\\n    if [item[\\"type\\"].rsplit(\\".\\", 1)[-1] for item in modules] != spec[\\"st_modules\\"]:\\n        raise ValueError(\\"Unexpected Sentence Transformers modules; a learned transform may be missing\\")\\n    pooling = json.loads((directory / \\"1_Pooling/config.json\\").read_text())\\n    intended = \\"pooling_mode_mean_tokens\\" if model_key == \\"styledistance\\" else \\"pooling_mode_lasttoken\\"\\n    if pooling.get(intended) is not True:\\n        raise ValueError(\\"Upstream pooling differs from the reviewed feature recipe\\")\\n    for key, enabled in pooling.items():\\n        if key.startswith(\\"pooling_mode_\\") and key != intended and enabled:\\n            raise ValueError(\\"Upstream uses additional pooling modes that must be implemented explicitly\\")\\n    return manifest\\n\\n\\ndef masked_mean_pool(hidden, attention_mask):\\n    \\"\\"\\"Native StyleDistance pooling counts every unmasked token, including specials.\\"\\"\\"\\n    import torch\\n\\n    mask = attention_mask.unsqueeze(-1).to(torch.float32)\\n    counts = mask.sum(dim=1)\\n    if torch.any(counts == 0):\\n        raise ValueError(\\"Cannot pool an empty token sequence\\")\\n    return (hidden.float() * mask).sum(dim=1) / counts\\n\\n\\ndef last_non_padding_pool(hidden, attention_mask):\\n    \\"\\"\\"Select the last actual token for either left- or right-padded batches.\\"\\"\\"\\n    import torch\\n\\n    if torch.any(attention_mask.sum(dim=1) == 0):\\n        raise ValueError(\\"Cannot pool an empty token sequence\\")\\n    positions = torch.arange(attention_mask.shape[1], device=attention_mask.device)\\n    last = positions.unsqueeze(0).expand_as(attention_mask).masked_fill(attention_mask == 0, -1).max(dim=1).values\\n    return hidden[torch.arange(hidden.shape[0], device=hidden.device), last].float()\\n\\n\\ndef extract_embeddings(\\n    texts: list[str], model_path: str | Path, model_key: str,\\n    *, batch_size: int = 16, max_length: int = 512, device: str = \\"cuda:0\\",\\n) -> tuple[np.ndarray, dict]:\\n    if model_key not in ENCODERS:\\n        raise ValueError(\\"Unknown encoder key\\")\\n    if batch_size < 1 or not 16 <= max_length <= 512:\\n        raise ValueError(\\"Use a positive batch size and a token limit from 16 to 512\\")\\n    if not texts or any(not isinstance(text, str) or not text.strip() for text in texts):\\n        raise ValueError(\\"Encoder inputs must be nonempty passage strings\\")\\n    spec = ENCODERS[model_key]\\n    validate_snapshot(model_path, model_key)\\n    import torch\\n    from transformers import AutoModel, AutoTokenizer\\n\\n    target = torch.device(device)\\n    if target.type not in {\\"cuda\\", \\"cpu\\"}:\\n        raise ValueError(\\"Use CUDA for the experiment or CPU for a small loading check\\")\\n    if target.type == \\"cuda\\":\\n        if not torch.cuda.is_available():\\n            raise ValueError(\\"CUDA unavailable; enable a free Kaggle GPU\\")\\n        _initialize_cuda_memory_stats(torch, target)\\n    started = time.perf_counter()\\n    tokenizer = AutoTokenizer.from_pretrained(\\n        str(model_path), local_files_only=True, trust_remote_code=False,\\n        padding_side=spec[\\"padding_side\\"], use_fast=True,\\n    )\\n    if tokenizer.pad_token_id is None:\\n        if tokenizer.eos_token_id is None:\\n            raise ValueError(\\"Encoder tokenizer needs a padding or EOS token\\")\\n        tokenizer.pad_token = tokenizer.eos_token\\n    # The complete embedding backbone is used. There is no layer truncation,\\n    # language-model output head, quantization, task instruction, or fine-tuning.\\n    model = AutoModel.from_pretrained(\\n        str(model_path), local_files_only=True, trust_remote_code=False,\\n        use_safetensors=True, dtype=torch.float32,\\n    ).to(target)\\n    model.config.use_cache = False\\n    model.requires_grad_(False)\\n    model.eval()\\n    if any(parameter.requires_grad for parameter in model.parameters()):\\n        raise ValueError(\\"The encoder must remain completely frozen\\")\\n    chunks, lengths = [], []\\n    try:\\n        with torch.inference_mode():\\n            for offset in range(0, len(texts), batch_size):\\n                passages = texts[offset:offset + batch_size]\\n                lengths.extend(tokenizer(passages, truncation=False, return_length=True)[\\"length\\"])\\n                batch = tokenizer(\\n                    passages, padding=True, truncation=True, max_length=max_length,\\n                    return_tensors=\\"pt\\",\\n                ).to(target)\\n                hidden = model(**batch).last_hidden_state\\n                if spec[\\"pooling\\"] == \\"last_non_padding_token\\":\\n                    vectors = last_non_padding_pool(hidden, batch[\\"attention_mask\\"])\\n                else:\\n                    vectors = masked_mean_pool(hidden, batch[\\"attention_mask\\"])\\n                norms = vectors.norm(p=2, dim=1, keepdim=True)\\n                if not torch.isfinite(vectors).all() or torch.any(norms <= 0):\\n                    raise ValueError(\\"Encoder produced non-finite or zero embeddings\\")\\n                vectors = vectors / norms\\n                chunks.append(vectors.cpu().numpy())\\n                del hidden, vectors, batch\\n        if target.type == \\"cuda\\":\\n            torch.cuda.synchronize(target)\\n        embeddings = np.concatenate(chunks).astype(np.float32, copy=False)\\n        if embeddings.shape != (len(texts), spec[\\"dimension\\"]):\\n            raise ValueError(\\"Unexpected encoder embedding shape\\")\\n        if not np.isfinite(embeddings).all() or not np.allclose(np.linalg.norm(embeddings, axis=1), 1, atol=1e-5):\\n            raise ValueError(\\"Frozen embeddings must be finite unit vectors\\")\\n        versions = {package: importlib.metadata.version(package) for package in (\\"torch\\", \\"transformers\\", \\"numpy\\")}\\n        metadata = {\\n            \\"model_id\\": spec[\\"model_id\\"], \\"model_revision\\": spec[\\"model_revision\\"],\\n            \\"encoder_frozen\\": True,\\n            \\"feature_options\\": {\\n                \\"pooling\\": spec[\\"pooling\\"], \\"max_length\\": max_length,\\n                \\"normalization\\": \\"l2\\", \\"padding_side\\": spec[\\"padding_side\\"],\\n                \\"compute_dtype\\": \\"float32\\", \\"instruction_prefix\\": \\"\\",\\n                \\"special_tokens_included\\": True, \\"quantization\\": \\"none\\",\\n                \\"full_backbone\\": True,\\n                \\"use_cache\\": False,\\n            },\\n            \\"profile\\": {\\n                \\"rows\\": len(texts), \\"dimension\\": embeddings.shape[1],\\n                \\"elapsed_seconds\\": time.perf_counter() - started,\\n                \\"device\\": str(target),\\n                \\"gpu\\": torch.cuda.get_device_name(target) if target.type == \\"cuda\\" else None,\\n                \\"peak_cuda_allocated_bytes\\": torch.cuda.max_memory_allocated(target) if target.type == \\"cuda\\" else None,\\n                \\"truncated_rows\\": int(sum(length > max_length for length in lengths)),\\n                \\"maximum_input_tokens\\": int(max(lengths)),\\n                \\"token_length_quantiles\\": {str(q): float(np.quantile(lengths, q)) for q in (0.5, 0.9, 0.99)},\\n                \\"versions\\": versions,\\n            },\\n        }\\n        return embeddings, metadata\\n    finally:\\n        del model\\n        gc.collect()\\n        if target.type == \\"cuda\\":\\n            torch.cuda.empty_cache()\\n\\n\\ndef main(argv: list[str] | None = None) -> int:\\n    parser = argparse.ArgumentParser(description=__doc__)\\n    parser.add_argument(\\"--train\\", type=Path, required=True)\\n    parser.add_argument(\\"--baseline-oof\\", type=Path, required=True)\\n    parser.add_argument(\\"--fold-manifest\\", type=Path, required=True)\\n    parser.add_argument(\\"--model-key\\", choices=tuple(ENCODERS), required=True)\\n    parser.add_argument(\\"--model-path\\", type=Path, required=True)\\n    parser.add_argument(\\"--output\\", type=Path, required=True)\\n    parser.add_argument(\\"--device\\", default=\\"cuda:0\\")\\n    parser.add_argument(\\"--batch-size\\", type=int, default=16)\\n    parser.add_argument(\\"--max-length\\", type=int, default=512)\\n    args = parser.parse_args(argv)\\n    if args.output.exists():\\n        raise ValueError(\\"Feature output already exists; preserve it or choose another output path\\")\\n    frame, protocol = prepare_gemini_evaluation(\\n        read_train(args.train), pd.read_parquet(args.baseline_oof), args.fold_manifest,\\n        OFFICIAL_BASELINE_COUNTS,\\n    )\\n    embeddings, metadata = extract_embeddings(\\n        frame[\\"text\\"].tolist(), args.model_path, args.model_key,\\n        batch_size=args.batch_size, max_length=args.max_length, device=args.device,\\n    )\\n    metadata[\\"train_fingerprint\\"] = protocol[\\"train_fingerprint\\"]\\n    metadata[\\"baseline_oof_fingerprint\\"] = protocol[\\"baseline_oof_fingerprint\\"]\\n    saved = save_feature_archive(\\n        args.output, frame[\\"id\\"].tolist(), embeddings, metadata, texts=frame[\\"text\\"].tolist(),\\n    )\\n    args.output.with_suffix(\\".json\\").write_text(json.dumps(saved, indent=2) + \\"\\\\n\\")\\n    print(json.dumps({\\"feature_archive\\": str(args.output), **metadata[\\"profile\\"]}, indent=2))\\n    return 0\\n\\n\\nif __name__ == \\"__main__\\":\\n    raise SystemExit(main())\\n", "ntl/features.py": "\\"\\"\\"Case-sensitive lexical features and transparent punctuation/style statistics.\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport re\\n\\nimport numpy as np\\nfrom sklearn.base import BaseEstimator, TransformerMixin\\nfrom sklearn.feature_extraction.text import TfidfVectorizer\\nfrom sklearn.pipeline import FeatureUnion, Pipeline\\nfrom sklearn.preprocessing import StandardScaler\\n\\n\\nclass StylometryTransformer(TransformerMixin, BaseEstimator):\\n    \\"\\"\\"Small numeric feature block; preserve the original punctuation/case.\\"\\"\\"\\n\\n    FEATURE_NAMES = (\\n        \\"log_char_count\\", \\"log_word_count\\", \\"mean_word_length\\", \\"mean_sentence_words\\",\\n        \\"uppercase_ratio\\", \\"digit_ratio\\", \\"whitespace_ratio\\", \\"newline_ratio\\",\\n        \\"comma_ratio\\", \\"semicolon_ratio\\", \\"colon_ratio\\", \\"exclamation_ratio\\",\\n        \\"question_ratio\\", \\"dash_ratio\\", \\"em_dash_ratio\\", \\"curly_quote_ratio\\",\\n        \\"straight_quote_ratio\\", \\"parenthesis_ratio\\", \\"markdown_ratio\\", \\"unique_word_ratio\\",\\n    )\\n\\n    def fit(self, X, y=None):\\n        return self\\n\\n    def transform(self, X):\\n        rows = []\\n        for text in X:\\n            words = re.findall(r\\"\\\\b\\\\w+\\\\b\\", text, flags=re.UNICODE)\\n            chars = max(len(text), 1)\\n            count = max(len(words), 1)\\n            sentences = max(len(re.findall(r\\"[.!?]+(?:\\\\s|$)\\", text)), 1)\\n            rows.append([\\n                np.log1p(len(text)), np.log1p(len(words)),\\n                sum(map(len, words)) / count, len(words) / sentences,\\n                sum(char.isupper() for char in text) / chars,\\n                sum(char.isdigit() for char in text) / chars,\\n                sum(char.isspace() for char in text) / chars,\\n                text.count(\\"\\\\n\\") / chars, text.count(\\",\\") / chars,\\n                text.count(\\";\\") / chars, text.count(\\":\\") / chars,\\n                text.count(\\"!\\") / chars, text.count(\\"?\\") / chars,\\n                text.count(\\"-\\") / chars, text.count(\\"\\u2014\\") / chars,\\n                sum(text.count(char) for char in \\"\\u2018\\u2019\\u201c\\u201d\\") / chars,\\n                sum(text.count(char) for char in \\"\\\\\\"\'\\") / chars,\\n                sum(text.count(char) for char in \\"()[]{}\\") / chars,\\n                sum(text.count(char) for char in \\"*#`_\\") / chars,\\n                len(set(words)) / count,\\n            ])\\n        return np.asarray(rows, dtype=np.float64).reshape(-1, len(self.FEATURE_NAMES))\\n\\n    def get_feature_names_out(self, input_features=None):\\n        return np.asarray(self.FEATURE_NAMES, dtype=object)\\n\\n\\ndef build_features(char_max_features: int = 50000, word_max_features: int = 30000) -> FeatureUnion:\\n    return FeatureUnion(\\n        [\\n            (\\"char\\", TfidfVectorizer(\\n                analyzer=\\"char\\", ngram_range=(3, 5), lowercase=False, strip_accents=None,\\n                max_features=char_max_features, min_df=2, sublinear_tf=True, dtype=np.float64,\\n            )),\\n            (\\"word\\", TfidfVectorizer(\\n                analyzer=\\"word\\", ngram_range=(1, 2), lowercase=False, strip_accents=None,\\n                token_pattern=r\\"(?u)\\\\b\\\\w+\\\\b\\", max_features=word_max_features,\\n                min_df=2, sublinear_tf=True, dtype=np.float64,\\n            )),\\n            (\\"style\\", Pipeline([\\n                (\\"extract\\", StylometryTransformer()),\\n                (\\"scale\\", StandardScaler(with_mean=False)),\\n            ])),\\n        ],\\n        transformer_weights={\\"char\\": 1.0, \\"word\\": 1.0, \\"style\\": 0.1},\\n        n_jobs=1,\\n    )\\n", "ntl/folds.py": "\\"\\"\\"Persisted, fingerprinted prompt-grouped cross-validation assignments.\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport json\\nfrom pathlib import Path\\n\\nimport numpy as np\\nimport pandas as pd\\nfrom sklearn.model_selection import StratifiedGroupKFold\\n\\nfrom .constants import DEFAULT_SEED, LABELS\\nfrom .data import fingerprint_frame, validate_train\\n\\n\\ndef build_fold_manifest(\\n    train: pd.DataFrame, n_splits: int = 5, seed: int = DEFAULT_SEED\\n) -> tuple[pd.DataFrame, dict]:\\n    train = validate_train(train)\\n    if n_splits < 2:\\n        raise ValueError(\\"Cross-validation requires at least two folds\\")\\n    if train[\\"prompt_id\\"].nunique() < n_splits:\\n        raise ValueError(\\"There are fewer unique prompts than requested folds\\")\\n    prompt_counts = train.groupby(\\"model\\")[\\"prompt_id\\"].nunique()\\n    if (prompt_counts < n_splits).any():\\n        raise ValueError(\\"Each model must appear in at least n_splits different prompts\\")\\n    # Sorting makes this stable even if a CSV is loaded in a different row order.\\n    ordered = train.sort_values([\\"prompt_id\\", \\"id\\"], kind=\\"stable\\").reset_index(drop=True)\\n    splitter = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)\\n    assignments = np.full(len(ordered), -1, dtype=np.int64)\\n    for fold, (_, valid_index) in enumerate(\\n        splitter.split(ordered[\\"text\\"], ordered[\\"model\\"], groups=ordered[\\"prompt_id\\"])\\n    ):\\n        assignments[valid_index] = fold\\n    manifest = ordered[[\\"id\\", \\"prompt_id\\"]].assign(fold=assignments).sort_values(\\"id\\").reset_index(drop=True)\\n    metadata = {\\n        \\"schema_version\\": 1,\\n        \\"algorithm\\": \\"StratifiedGroupKFold\\",\\n        \\"group_column\\": \\"prompt_id\\",\\n        \\"n_splits\\": n_splits,\\n        \\"seed\\": seed,\\n        \\"train_fingerprint\\": fingerprint_frame(train),\\n        \\"n_rows\\": len(train),\\n        \\"n_prompts\\": int(train[\\"prompt_id\\"].nunique()),\\n    }\\n    validate_fold_manifest(train, manifest, metadata)\\n    return manifest, metadata\\n\\n\\ndef validate_fold_manifest(train: pd.DataFrame, manifest: pd.DataFrame, metadata: dict) -> None:\\n    train = validate_train(train)\\n    if metadata.get(\\"schema_version\\") != 1 or metadata.get(\\"group_column\\") != \\"prompt_id\\":\\n        raise ValueError(\\"Unsupported fold manifest schema or grouping column\\")\\n    if metadata.get(\\"train_fingerprint\\") != fingerprint_frame(train):\\n        raise ValueError(\\"Fold manifest fingerprint differs from the training data\\")\\n    if set(manifest.columns) != {\\"id\\", \\"prompt_id\\", \\"fold\\"}:\\n        raise ValueError(\\"Fold manifest must have id, prompt_id and fold columns\\")\\n    if manifest.isna().any().any() or manifest[\\"id\\"].duplicated().any():\\n        raise ValueError(\\"Fold manifest contains nulls or duplicate ids\\")\\n    if len(manifest) != len(train) or set(manifest[\\"id\\"]) != set(train[\\"id\\"]):\\n        raise ValueError(\\"Fold manifest must cover every training id exactly once\\")\\n    n_splits = metadata.get(\\"n_splits\\")\\n    if not isinstance(n_splits, int) or n_splits < 2:\\n        raise ValueError(\\"Invalid number of folds in manifest metadata\\")\\n    fold_values = manifest[\\"fold\\"]\\n    if not pd.api.types.is_integer_dtype(fold_values):\\n        raise ValueError(\\"Fold assignments must be integers\\")\\n    if set(fold_values) != set(range(n_splits)):\\n        raise ValueError(\\"Fold assignments must cover exactly 0 through n_splits - 1\\")\\n    aligned = train.merge(manifest, on=\\"id\\", suffixes=(\\"\\", \\"_manifest\\"), validate=\\"one_to_one\\")\\n    if not (aligned[\\"prompt_id\\"] == aligned[\\"prompt_id_manifest\\"]).all():\\n        raise ValueError(\\"Fold manifest prompt ids do not match the training data\\")\\n    if aligned.groupby(\\"prompt_id\\")[\\"fold\\"].nunique().max() != 1:\\n        raise ValueError(\\"Prompt leakage: a prompt occurs in more than one fold\\")\\n    for fold in range(n_splits):\\n        fit = aligned.loc[aligned[\\"fold\\"] != fold]\\n        valid = aligned.loc[aligned[\\"fold\\"] == fold]\\n        if set(fit[\\"prompt_id\\"]) & set(valid[\\"prompt_id\\"]):\\n            raise ValueError(\\"Prompt leakage between fitting and validation rows\\")\\n        if set(fit[\\"model\\"]) != set(LABELS) or set(valid[\\"model\\"]) != set(LABELS):\\n            raise ValueError(f\\"Fold {fold} does not include all labels in both training and validation\\")\\n\\n\\ndef save_fold_manifest(manifest: pd.DataFrame, metadata: dict, directory: str | Path) -> Path:\\n    directory = Path(directory)\\n    directory.mkdir(parents=True, exist_ok=True)\\n    manifest.to_csv(directory / \\"folds.csv\\", index=False)\\n    (directory / \\"folds.json\\").write_text(json.dumps(metadata, indent=2) + \\"\\\\n\\")\\n    return directory / \\"folds.csv\\"\\n\\n\\ndef load_fold_manifest(train: pd.DataFrame, path: str | Path) -> tuple[pd.DataFrame, dict]:\\n    path = Path(path)\\n    if path.is_dir():\\n        path = path / \\"folds.csv\\"\\n    manifest = pd.read_csv(path, dtype={\\"id\\": str, \\"prompt_id\\": str}, keep_default_na=False)\\n    metadata = json.loads(path.with_suffix(\\".json\\").read_text())\\n    validate_fold_manifest(train, manifest, metadata)\\n    return manifest, metadata\\n", "ntl/frozen_gemini.py": "\\"\\"\\"Nested prompt-grouped linear readouts of immutable frozen Gemini embeddings.\\n\\nEncoder extraction is independent of labels. All learned scaling and regularized\\nclassification are fit within the inner/outer training partitions. The original\\n570-row CPU-routed cohort changes reporting only; all 600 true-Gemini rows enter\\nthe original five outer prompt folds.\\n\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport argparse\\nimport hashlib\\nimport importlib.metadata\\nimport json\\nimport re\\nimport time\\nfrom dataclasses import asdict, dataclass\\nfrom pathlib import Path\\nfrom typing import Sequence\\n\\nimport numpy as np\\nimport pandas as pd\\nfrom sklearn.linear_model import LogisticRegression\\nfrom sklearn.metrics import balanced_accuracy_score\\nfrom sklearn.model_selection import StratifiedGroupKFold\\nfrom sklearn.pipeline import Pipeline\\nfrom sklearn.preprocessing import StandardScaler\\nfrom threadpoolctl import threadpool_limits\\n\\nfrom .data import read_train\\nfrom .gemini import (\\n    GEMINI_LABELS,\\n    OFFICIAL_BASELINE_COUNTS,\\n    _cohort_metrics,\\n    _save_class_reports,\\n    prepare_gemini_evaluation,\\n)\\n\\n\\n@dataclass(frozen=True)\\nclass FrozenGeminiConfig:\\n    C_grid: tuple[float, ...] = (0.001, 0.01, 0.1, 1.0, 10.0)\\n    inner_folds: int = 3\\n    max_iter: int = 2000\\n    seed: int = 2026\\n\\n    def validate(self) -> None:\\n        if not self.C_grid or any(not np.isfinite(c) or c <= 0 for c in self.C_grid):\\n            raise ValueError(\\"Regularization grid must contain finite positive C values\\")\\n        if len(set(self.C_grid)) != len(self.C_grid):\\n            raise ValueError(\\"Regularization grid contains duplicate C values\\")\\n        if self.inner_folds < 2 or self.max_iter < 1:\\n            raise ValueError(\\"Inner fold count must be at least two and max_iter must be positive\\")\\n\\n\\ndef _json(path: Path, value: dict | list) -> None:\\n    path.write_text(json.dumps(value, indent=2, allow_nan=False) + \\"\\\\n\\")\\n\\n\\ndef _string_ids(ids: Sequence[str]) -> list[str]:\\n    values = list(ids)\\n    if not values or any(not isinstance(value, str) or not value.strip() for value in values):\\n        raise ValueError(\\"Embedding ids must be nonempty strings\\")\\n    if len(set(values)) != len(values):\\n        raise ValueError(\\"Embedding ids must be unique\\")\\n    return values\\n\\n\\ndef ids_sha256(ids: Sequence[str]) -> str:\\n    \\"\\"\\"SHA-256 of UTF-8 compact JSON list, retaining the archive\'s row order.\\"\\"\\"\\n    return hashlib.sha256(json.dumps(_string_ids(ids), ensure_ascii=False, separators=(\\",\\", \\":\\")).encode()).hexdigest()\\n\\n\\ndef embeddings_sha256(features: np.ndarray) -> str:\\n    \\"\\"\\"Hash sorted-key JSON shape/dtype.str followed by C-order raw array bytes.\\n\\n    Archive dtype and byte order are preserved; this does not round, cast or\\n    normalize values. Import this helper instead of duplicating the contract.\\n    \\"\\"\\"\\n    array = np.asarray(features)\\n    if array.ndim != 2 or not np.issubdtype(array.dtype, np.floating):\\n        raise ValueError(\\"Embeddings must be a two-dimensional floating-point array\\")\\n    if min(array.shape) < 1 or not np.isfinite(array).all():\\n        raise ValueError(\\"Embeddings must be nonempty and finite\\")\\n    digest = hashlib.sha256()\\n    digest.update(json.dumps({\\"shape\\": list(array.shape), \\"dtype\\": array.dtype.str}, sort_keys=True).encode())\\n    digest.update(np.ascontiguousarray(array).tobytes(order=\\"C\\"))\\n    return digest.hexdigest()\\n\\n\\ndef input_text_sha256(ids: Sequence[str], texts: Sequence[str]) -> str:\\n    \\"\\"\\"Hash compact UTF-8 JSON [id, exact_text] lines in archive row order.\\"\\"\\"\\n    ids = _string_ids(ids)\\n    texts = list(texts)\\n    if len(ids) != len(texts) or any(not isinstance(text, str) or not text.strip() for text in texts):\\n        raise ValueError(\\"Each embedding id needs its exact nonempty input text\\")\\n    digest = hashlib.sha256()\\n    for pair in zip(ids, texts, strict=True):\\n        digest.update(json.dumps(pair, ensure_ascii=False, separators=(\\",\\", \\":\\")).encode())\\n        digest.update(b\\"\\\\n\\")\\n    return digest.hexdigest()\\n\\n\\ndef _validate_provenance(metadata: dict) -> None:\\n    if metadata.get(\\"schema_version\\") != 1 or metadata.get(\\"encoder_frozen\\") is not True:\\n        raise ValueError(\\"Features require schema_version 1 and encoder_frozen=true provenance\\")\\n    if not isinstance(metadata.get(\\"model_id\\"), str) or not metadata[\\"model_id\\"].strip():\\n        raise ValueError(\\"Feature provenance is missing model_id\\")\\n    if not isinstance(metadata.get(\\"model_revision\\"), str) or not re.fullmatch(r\\"[0-9a-f]{40}\\", metadata[\\"model_revision\\"]):\\n        raise ValueError(\\"Feature provenance requires an immutable model revision\\")\\n    options = metadata.get(\\"feature_options\\")\\n    if not isinstance(options, dict) or not {\\"pooling\\", \\"max_length\\", \\"normalization\\"}.issubset(options):\\n        raise ValueError(\\"Feature provenance must describe pooling, max_length and normalization\\")\\n    if not isinstance(options[\\"pooling\\"], str) or not options[\\"pooling\\"].strip():\\n        raise ValueError(\\"Feature pooling must be explicit\\")\\n    if not isinstance(options[\\"max_length\\"], int) or options[\\"max_length\\"] < 1:\\n        raise ValueError(\\"Feature max_length must be a positive integer\\")\\n    if not isinstance(options[\\"normalization\\"], (str, bool)):\\n        raise ValueError(\\"Fixed feature normalization must be explicit\\")\\n\\n\\ndef save_feature_archive(\\n    path: str | Path, ids: Sequence[str], embeddings: np.ndarray, metadata: dict,\\n    *, texts: Sequence[str],\\n) -> dict:\\n    \\"\\"\\"Write a pickle-free NPZ with exact row/text/array identity and source pins.\\"\\"\\"\\n    ids = _string_ids(ids)\\n    array = np.asarray(embeddings)\\n    array_hash = embeddings_sha256(array)\\n    if len(ids) != len(array):\\n        raise ValueError(\\"Embedding row count does not match ids\\")\\n    value = dict(metadata)\\n    value[\\"schema_version\\"] = 1\\n    value[\\"ids_sha256\\"] = ids_sha256(ids)\\n    value[\\"embeddings_sha256\\"] = array_hash\\n    value[\\"input_text_sha256\\"] = input_text_sha256(ids, texts)\\n    _validate_provenance(value)\\n    destination = Path(path)\\n    if destination.suffix != \\".npz\\":\\n        raise ValueError(\\"Embedding archive filename must end in .npz\\")\\n    destination.parent.mkdir(parents=True, exist_ok=True)\\n    np.savez_compressed(destination, ids=np.asarray(ids, dtype=str), embeddings=array,\\n                        metadata=np.asarray(json.dumps(value, sort_keys=True)))\\n    return value\\n\\n\\ndef load_feature_archive(path: str | Path, gemini_frame: pd.DataFrame) -> tuple[np.ndarray, dict]:\\n    \\"\\"\\"Verify stored row order before aligning features to the evaluation frame.\\"\\"\\"\\n    path = Path(path)\\n    with np.load(path, allow_pickle=False) as archive:\\n        if set(archive.files) != {\\"ids\\", \\"embeddings\\", \\"metadata\\"}:\\n            raise ValueError(\\"Feature archive must contain exactly ids, embeddings and metadata\\")\\n        ids_array = archive[\\"ids\\"]\\n        if ids_array.ndim != 1 or ids_array.dtype.kind != \\"U\\":\\n            raise ValueError(\\"Feature ids must be a one-dimensional Unicode array\\")\\n        ids = _string_ids(ids_array.tolist())\\n        features = archive[\\"embeddings\\"]\\n        encoded_metadata = archive[\\"metadata\\"]\\n        if encoded_metadata.ndim != 0 or encoded_metadata.dtype.kind != \\"U\\":\\n            raise ValueError(\\"Feature metadata must be a scalar JSON string\\")\\n        metadata = json.loads(str(encoded_metadata.item()))\\n    if not isinstance(metadata, dict):\\n        raise ValueError(\\"Feature metadata must be a JSON object\\")\\n    _validate_provenance(metadata)\\n    if metadata.get(\\"ids_sha256\\") != ids_sha256(ids):\\n        raise ValueError(\\"Feature id hash mismatch\\")\\n    if metadata.get(\\"embeddings_sha256\\") != embeddings_sha256(features):\\n        raise ValueError(\\"Feature array hash mismatch\\")\\n    if len(features) != len(ids):\\n        raise ValueError(\\"Embedding rows and ids differ in length\\")\\n    expected_ids = _string_ids(gemini_frame[\\"id\\"].tolist())\\n    if len(ids) != len(expected_ids) or set(ids) != set(expected_ids):\\n        raise ValueError(\\"Feature ids must exactly cover all true-Gemini evaluation rows\\")\\n    texts = gemini_frame.set_index(\\"id\\")[\\"text\\"]\\n    if metadata.get(\\"input_text_sha256\\") != input_text_sha256(ids, texts.loc[ids].tolist()):\\n        raise ValueError(\\"Feature input-text hash mismatch\\")\\n    row_lookup = {value: index for index, value in enumerate(ids)}\\n    aligned = features[[row_lookup[value] for value in expected_ids]]\\n    provenance = dict(metadata)\\n    provenance[\\"archive_sha256\\"] = hashlib.sha256(path.read_bytes()).hexdigest()\\n    provenance[\\"aligned_embeddings_sha256\\"] = embeddings_sha256(aligned)\\n    return aligned, provenance\\n\\n\\ndef _validate_frame(frame: pd.DataFrame) -> pd.DataFrame:\\n    required = {\\"id\\", \\"prompt_id\\", \\"model\\", \\"fold\\", \\"baseline_model\\", \\"selected\\", \\"baseline_correct\\"}\\n    if not required.issubset(frame.columns) or frame.empty:\\n        raise ValueError(\\"Gemini evaluation frame is missing required data\\")\\n    if frame[list(required)].isna().any().any() or frame[\\"id\\"].duplicated().any():\\n        raise ValueError(\\"Gemini evaluation rows contain nulls or repeated ids\\")\\n    _string_ids(frame[\\"id\\"].tolist())\\n    if set(frame[\\"model\\"]) != set(GEMINI_LABELS):\\n        raise ValueError(\\"Gemini evaluation frame must contain all four canonical Gemini labels\\")\\n    if not pd.api.types.is_integer_dtype(frame[\\"fold\\"]) or set(frame[\\"fold\\"]) != set(range(5)):\\n        raise ValueError(\\"Gemini evaluation requires the original five outer folds\\")\\n    if frame.groupby(\\"prompt_id\\")[\\"fold\\"].nunique().max() != 1:\\n        raise ValueError(\\"Prompt leakage between outer folds\\")\\n    if not pd.api.types.is_bool_dtype(frame[\\"selected\\"]) or not pd.api.types.is_bool_dtype(frame[\\"baseline_correct\\"]):\\n        raise ValueError(\\"Cohort and baseline correctness masks must be boolean\\")\\n    if not frame[\\"selected\\"].equals(frame[\\"baseline_model\\"].isin(GEMINI_LABELS)):\\n        raise ValueError(\\"Selected cohort differs from the saved Gemini-routing condition\\")\\n    if not frame[\\"baseline_correct\\"].equals(frame[\\"baseline_model\\"] == frame[\\"model\\"]):\\n        raise ValueError(\\"Baseline correctness mask differs from the true labels\\")\\n    if not frame[\\"selected\\"].any():\\n        raise ValueError(\\"The selected Gemini reporting cohort is empty\\")\\n    return frame.reset_index(drop=True).copy()\\n\\n\\ndef _pipeline(C: float, config: FrozenGeminiConfig) -> Pipeline:\\n    return Pipeline([\\n        (\\"scale\\", StandardScaler()),\\n        (\\"classifier\\", LogisticRegression(C=C, penalty=\\"l2\\", solver=\\"lbfgs\\",\\n                                           max_iter=config.max_iter, random_state=config.seed)),\\n    ])\\n\\n\\ndef _aligned_probabilities(pipeline: Pipeline, features: np.ndarray) -> np.ndarray:\\n    classes = list(pipeline.named_steps[\\"classifier\\"].classes_)\\n    if len(classes) != 4 or set(classes) != set(range(4)):\\n        raise ValueError(\\"Classifier did not fit the four canonical Gemini label indices\\")\\n    values = pipeline.predict_proba(features)[:, [classes.index(index) for index in range(4)]]\\n    if not np.isfinite(values).all() or (values < 0).any() or not np.allclose(values.sum(axis=1), 1.0):\\n        raise ValueError(\\"Classifier probabilities are non-finite or invalid\\")\\n    return values\\n\\n\\ndef _select_regularization(\\n    fitting_features: np.ndarray, fitting_targets: np.ndarray, fitting_prompts: np.ndarray,\\n    config: FrozenGeminiConfig, outer_fold: int,\\n) -> tuple[float, dict]:\\n    \\"\\"\\"No outer-validation features, labels or scores enter this selection API.\\"\\"\\"\\n    splitter = StratifiedGroupKFold(n_splits=config.inner_folds, shuffle=True,\\n                                    random_state=config.seed + outer_fold)\\n    splits = list(splitter.split(fitting_features, fitting_targets, groups=fitting_prompts))\\n    split_metadata = []\\n    for inner_fold, (fit, valid) in enumerate(splits):\\n        fit_groups, valid_groups = set(fitting_prompts[fit]), set(fitting_prompts[valid])\\n        if fit_groups & valid_groups:\\n            raise ValueError(\\"Prompt leakage in inner regularization folds\\")\\n        if set(fitting_targets[fit]) != set(range(4)) or set(fitting_targets[valid]) != set(range(4)):\\n            raise ValueError(\\"Every inner fitting/validation partition must contain all four Gemini classes\\")\\n        split_metadata.append({\\"inner_fold\\": inner_fold, \\"training_rows\\": len(fit),\\n                               \\"validation_rows\\": len(valid), \\"training_prompts\\": sorted(fit_groups),\\n                               \\"validation_prompts\\": sorted(valid_groups)})\\n    candidates = []\\n    for C in sorted(config.C_grid):\\n        scores = []\\n        for fit, valid in splits:\\n            estimator = _pipeline(C, config)\\n            estimator.fit(fitting_features[fit], fitting_targets[fit])\\n            probabilities = _aligned_probabilities(estimator, fitting_features[valid])\\n            scores.append(float(balanced_accuracy_score(fitting_targets[valid], probabilities.argmax(axis=1))))\\n        candidates.append({\\"C\\": C, \\"inner_balanced_accuracy\\": scores,\\n                           \\"mean_inner_balanced_accuracy\\": float(np.mean(scores))})\\n    best = min(candidates, key=lambda item: (-item[\\"mean_inner_balanced_accuracy\\"], item[\\"C\\"]))\\n    return best[\\"C\\"], {\\"outer_fold\\": outer_fold, \\"selected_C\\": best[\\"C\\"], \\"candidates\\": candidates,\\n                       \\"inner_splits\\": split_metadata,\\n                       \\"selection_metric\\": \\"Unweighted mean inner-fold balanced accuracy; ties choose smaller C.\\"}\\n\\n\\ndef evaluate_embeddings(\\n    gemini_frame: pd.DataFrame, features: np.ndarray, output_dir: str | Path,\\n    config: FrozenGeminiConfig | None = None, feature_metadata: dict | None = None,\\n    protocol_metadata: dict | None = None,\\n) -> dict:\\n    config = config or FrozenGeminiConfig()\\n    config.validate()\\n    frame = _validate_frame(gemini_frame)\\n    features = np.asarray(features)\\n    feature_hash = embeddings_sha256(features)\\n    if len(features) != len(frame):\\n        raise ValueError(\\"Features must already be aligned to every Gemini evaluation row\\")\\n    if feature_metadata is not None:\\n        _validate_provenance(feature_metadata)\\n        if feature_metadata.get(\\"aligned_embeddings_sha256\\", feature_metadata.get(\\"embeddings_sha256\\")) != feature_hash:\\n            raise ValueError(\\"Aligned feature hash differs from the provided provenance\\")\\n    output = Path(output_dir)\\n    output.mkdir(parents=True, exist_ok=True)\\n    if (output / \\"metrics.json\\").exists() or (output / \\"oof.parquet\\").exists():\\n        raise ValueError(\\"Use a new output directory for each frozen-encoder evaluation\\")\\n    targets = frame[\\"model\\"].map({label: index for index, label in enumerate(GEMINI_LABELS)}).to_numpy()\\n    probabilities = np.full((len(frame), 4), np.nan)\\n    counts = np.zeros(len(frame), dtype=np.int64)\\n    nested_results = []\\n    started = time.perf_counter()\\n    for outer_fold in range(5):\\n        fit, valid = (frame[\\"fold\\"] != outer_fold).to_numpy(), (frame[\\"fold\\"] == outer_fold).to_numpy()\\n        if set(frame.loc[fit, \\"prompt_id\\"]) & set(frame.loc[valid, \\"prompt_id\\"]):\\n            raise ValueError(\\"Prompt leakage detected before the outer classifier fit\\")\\n        if set(targets[fit]) != set(range(4)) or set(targets[valid]) != set(range(4)):\\n            raise ValueError(\\"Every outer fitting/validation partition must contain all four Gemini classes\\")\\n        with threadpool_limits(limits=2):\\n            C, nested = _select_regularization(features[fit], targets[fit], frame.loc[fit, \\"prompt_id\\"].to_numpy(),\\n                                               config, outer_fold)\\n            estimator = _pipeline(C, config)\\n            estimator.fit(features[fit], targets[fit])\\n            probabilities[valid] = _aligned_probabilities(estimator, features[valid])\\n        counts[valid] += 1\\n        nested[\\"outer_train_rows\\"] = int(fit.sum())\\n        nested[\\"outer_validation_rows\\"] = int(valid.sum())\\n        nested[\\"outer_train_selected_rows\\"] = int(frame.loc[fit, \\"selected\\"].sum())\\n        nested[\\"outer_validation_balanced_accuracy\\"] = float(balanced_accuracy_score(targets[valid], probabilities[valid].argmax(axis=1)))\\n        nested_results.append(nested)\\n        _json(output / \\"inner_cv.json\\", nested_results)\\n    if not np.all(counts == 1) or not np.isfinite(probabilities).all():\\n        raise ValueError(\\"Every true-Gemini row must have exactly one complete outer OOF prediction\\")\\n    result = frame[[\\"id\\", \\"prompt_id\\", \\"model\\", \\"fold\\", \\"baseline_model\\", \\"selected\\", \\"baseline_correct\\"]].copy()\\n    result[\\"prediction\\"] = np.asarray(GEMINI_LABELS)[probabilities.argmax(axis=1)]\\n    result[\\"confidence\\"] = probabilities.max(axis=1)\\n    result[\\"specialist_correct\\"] = result[\\"prediction\\"] == result[\\"model\\"]\\n    for index, label in enumerate(GEMINI_LABELS):\\n        result[f\\"probability::{label}\\"] = probabilities[:, index]\\n    selected = result.loc[result[\\"selected\\"]].copy()\\n    metrics = {\\"all_true_gemini\\": _cohort_metrics(result), \\"correctly_routed_gemini\\": _cohort_metrics(selected),\\n               \\"selected_C_per_outer_fold\\": [item[\\"selected_C\\"] for item in nested_results],\\n               \\"elapsed_seconds\\": time.perf_counter() - started,\\n               \\"interpretation\\": \\"Nested prompt-grouped frozen-feature comparison; 570-row selection is for reporting only, and is not a hidden-test or end-to-end routing score.\\"}\\n    versions = {package: importlib.metadata.version(package) for package in\\n                (\\"numpy\\", \\"pandas\\", \\"scikit-learn\\", \\"scipy\\")}\\n    protocol = None if protocol_metadata is None else dict(protocol_metadata)\\n    if protocol is not None:\\n        protocol[\\"outer_validation_policy\\"] = \\"C is selected only by inner prompt-grouped balanced accuracy; each outer validation partition is predicted once.\\"\\n    metadata = {\\"schema_version\\": 1, \\"kind\\": \\"frozen-encoder-nested-gemini-linear-readout\\",\\n                \\"labels\\": list(GEMINI_LABELS), \\"configuration\\": asdict(config), \\"versions\\": versions,\\n                \\"feature_provenance\\": feature_metadata,\\n                \\"aligned_embeddings_sha256\\": feature_hash,\\n                \\"aligned_ids_sha256\\": ids_sha256(frame[\\"id\\"].tolist()),\\n                \\"protocol\\": protocol,\\n                \\"selection_policy\\": \\"Inner prompt-grouped folds only; outer validation is predicted once after choosing C.\\",\\n                \\"training_population\\": \\"Every true-Gemini row in outer fitting folds, including CPU routing false negatives.\\",\\n                \\"learned_transforms\\": \\"Fresh StandardScaler and L2 logistic regression fit on each inner/outer fitting partition.\\"}\\n    result.to_parquet(output / \\"oof.parquet\\", index=False)\\n    selected.to_parquet(output / \\"selected_oof.parquet\\", index=False)\\n    selected.to_csv(output / \\"selected_predictions.csv\\", index=False)\\n    _save_class_reports(result, output, \\"\\")\\n    _save_class_reports(selected, output, \\"_selected\\")\\n    _json(output / \\"metadata.json\\", metadata)\\n    _json(output / \\"metrics.json\\", metrics)\\n    return metrics\\n\\n\\ndef main(argv: list[str] | None = None) -> int:\\n    parser = argparse.ArgumentParser(description=__doc__)\\n    parser.add_argument(\\"--train\\", type=Path, required=True)\\n    parser.add_argument(\\"--baseline-oof\\", type=Path, required=True)\\n    parser.add_argument(\\"--fold-manifest\\", type=Path, required=True)\\n    parser.add_argument(\\"--features\\", type=Path, required=True)\\n    parser.add_argument(\\"--output-dir\\", type=Path, required=True)\\n    parser.add_argument(\\"--C-grid\\", type=float, nargs=\\"+\\", default=[0.001, 0.01, 0.1, 1.0, 10.0])\\n    parser.add_argument(\\"--inner-folds\\", type=int, default=3)\\n    parser.add_argument(\\"--max-iter\\", type=int, default=2000)\\n    parser.add_argument(\\"--seed\\", type=int, default=2026)\\n    args = parser.parse_args(argv)\\n    frame, protocol = prepare_gemini_evaluation(read_train(args.train), pd.read_parquet(args.baseline_oof),\\n                                               args.fold_manifest, OFFICIAL_BASELINE_COUNTS)\\n    features, metadata = load_feature_archive(args.features, frame)\\n    config = FrozenGeminiConfig(C_grid=tuple(args.C_grid), inner_folds=args.inner_folds,\\n                                max_iter=args.max_iter, seed=args.seed)\\n    metrics = evaluate_embeddings(frame, features, args.output_dir, config, metadata, protocol)\\n    print(json.dumps(metrics, indent=2))\\n    return 0\\n\\n\\nif __name__ == \\"__main__\\":\\n    raise SystemExit(main())\\n", "ntl/gemini.py": "\\"\\"\\"A fixed, prompt-grouped DeBERTa experiment for exact Gemini version attribution.\\n\\nTrain each fresh four-class head/backbone on every true-Gemini training-fold row.\\nEvaluate all 600 held-out Gemini rows and the preselected 570 correctly routed by\\nthe saved CPU out-of-fold predictions. Outer validation never selects an epoch.\\n\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport argparse\\nimport gc\\nimport hashlib\\nimport importlib.metadata\\nimport json\\nimport math\\nimport random\\nimport time\\nfrom dataclasses import asdict, dataclass\\nfrom pathlib import Path\\n\\nimport numpy as np\\nimport pandas as pd\\nfrom sklearn.metrics import (\\n    accuracy_score,\\n    balanced_accuracy_score,\\n    confusion_matrix,\\n    precision_recall_fscore_support,\\n)\\n\\nfrom .constants import LABELS\\nfrom .data import fingerprint_frame, read_train, validate_train\\nfrom .folds import load_fold_manifest, save_fold_manifest\\n\\nMODEL_ID = \\"microsoft/deberta-v3-base\\"\\nMODEL_REVISION = \\"8ccc9b6f36199bec6961081d44eb72fb3f7353f3\\"\\nGEMINI_LABELS = tuple(label for label in LABELS if label.startswith(\\"gemini-\\"))\\nOFFICIAL_BASELINE_COUNTS = {\\n    \\"train_rows\\": 2699,\\n    \\"gemini_rows\\": 600,\\n    \\"selected_rows\\": 570,\\n    \\"baseline_selected_correct\\": 291,\\n}\\n\\n\\n@dataclass(frozen=True)\\nclass GeminiConfig:\\n    epochs: int = 3\\n    learning_rate: float = 2e-5\\n    max_length: int = 512\\n    batch_size: int = 8\\n    gradient_accumulation: int = 2\\n    warmup_ratio: float = 0.1\\n    weight_decay: float = 0.01\\n    max_grad_norm: float = 1.0\\n    amp_initial_scale: float = 1024.0\\n    max_consecutive_amp_skips: int = 20\\n    seed: int = 2026\\n    device: str = \\"cuda:0\\"\\n\\n    def validate(self) -> None:\\n        if self.epochs < 1 or self.batch_size < 1 or self.gradient_accumulation < 1:\\n            raise ValueError(\\"Epochs, microbatch size and accumulation must be positive\\")\\n        if not 16 <= self.max_length <= 512:\\n            raise ValueError(\\"The selected DeBERTa protocol requires a token limit from 16 to 512\\")\\n        if self.learning_rate <= 0 or self.max_grad_norm <= 0 or self.weight_decay < 0:\\n            raise ValueError(\\"Invalid optimizer configuration\\")\\n        if not 0 <= self.warmup_ratio < 1:\\n            raise ValueError(\\"Warmup ratio must be in [0, 1)\\")\\n        if self.amp_initial_scale <= 0 or self.max_consecutive_amp_skips < 1:\\n            raise ValueError(\\"Invalid AMP loss-scale recovery settings\\")\\n\\n\\ndef _json(path: Path, value: dict | list) -> None:\\n    path.write_text(json.dumps(value, indent=2, allow_nan=False) + \\"\\\\n\\")\\n\\n\\ndef _frame_digest(frame: pd.DataFrame, columns: list[str]) -> str:\\n    digest = hashlib.sha256()\\n    for row in frame.sort_values(\\"id\\").loc[:, columns].itertuples(index=False, name=None):\\n        digest.update(json.dumps(row, ensure_ascii=False, separators=(\\",\\", \\":\\")).encode())\\n        digest.update(b\\"\\\\n\\")\\n    return digest.hexdigest()\\n\\n\\ndef prepare_gemini_evaluation(\\n    train: pd.DataFrame,\\n    baseline_oof: pd.DataFrame,\\n    fold_manifest_path: str | Path,\\n    expected_counts: dict[str, int] | None = None,\\n) -> tuple[pd.DataFrame, dict]:\\n    \\"\\"\\"Join exact baseline OOF truth/IDs/folds before selecting the fixed cohort.\\n\\n    Selection changes the reporting cohort only, never the training population.\\n    The fingerprinted fold manifest always belongs to the complete 18-class train.\\n    \\"\\"\\"\\n    train = validate_train(train)\\n    manifest, fold_metadata = load_fold_manifest(train, fold_manifest_path)\\n    if fold_metadata[\\"n_splits\\"] != 5:\\n        raise ValueError(\\"This fixed comparison requires the original five prompt-grouped folds\\")\\n    required = {\\"id\\", \\"prompt_id\\", \\"true_model\\", \\"fold\\", \\"predicted_model\\"}\\n    if not required.issubset(baseline_oof.columns):\\n        raise ValueError(f\\"Baseline OOF is missing columns: {sorted(required - set(baseline_oof.columns))}\\")\\n    baseline = baseline_oof.loc[:, sorted(required)].copy()\\n    if baseline.isna().any().any() or baseline[\\"id\\"].duplicated().any():\\n        raise ValueError(\\"Baseline OOF contains null values or duplicate ids\\")\\n    for column in (\\"id\\", \\"prompt_id\\", \\"true_model\\", \\"predicted_model\\"):\\n        if not baseline[column].map(lambda value: isinstance(value, str) and bool(value.strip())).all():\\n            raise ValueError(f\\"Baseline OOF {column} must contain nonempty strings\\")\\n    if not pd.api.types.is_integer_dtype(baseline[\\"fold\\"]):\\n        raise ValueError(\\"Baseline OOF fold assignments must be integers\\")\\n    if len(baseline) != len(train) or set(baseline[\\"id\\"]) != set(train[\\"id\\"]):\\n        raise ValueError(\\"Baseline OOF must contain every official training id exactly once\\")\\n    if not set(baseline[\\"predicted_model\\"]).issubset(LABELS):\\n        raise ValueError(\\"Baseline OOF contains predictions outside the 18 valid labels\\")\\n    merged = train.merge(baseline, on=\\"id\\", suffixes=(\\"\\", \\"_baseline\\"), validate=\\"one_to_one\\")\\n    expected_folds = merged[\\"id\\"].map(manifest.set_index(\\"id\\")[\\"fold\\"])\\n    if not (merged[\\"prompt_id\\"] == merged[\\"prompt_id_baseline\\"]).all():\\n        raise ValueError(\\"Baseline OOF prompt ids differ from official training rows\\")\\n    if not (merged[\\"model\\"] == merged[\\"true_model\\"]).all():\\n        raise ValueError(\\"Baseline OOF true model labels differ from official training rows\\")\\n    if not (merged[\\"fold\\"] == expected_folds).all():\\n        raise ValueError(\\"Baseline OOF folds differ from the saved prompt-grouped manifest\\")\\n    gemini = merged.loc[merged[\\"model\\"].isin(GEMINI_LABELS)].copy()\\n    gemini = gemini.rename(columns={\\"predicted_model\\": \\"baseline_model\\"})\\n    gemini[\\"selected\\"] = gemini[\\"baseline_model\\"].isin(GEMINI_LABELS)\\n    gemini[\\"baseline_correct\\"] = gemini[\\"baseline_model\\"] == gemini[\\"model\\"]\\n    gemini[\\"target\\"] = gemini[\\"model\\"].map({label: index for index, label in enumerate(GEMINI_LABELS)})\\n    gemini = gemini.sort_values(\\"id\\", kind=\\"stable\\").reset_index(drop=True)\\n    for fold in range(5):\\n        fitting = gemini.loc[gemini[\\"fold\\"] != fold]\\n        validation = gemini.loc[gemini[\\"fold\\"] == fold]\\n        if set(fitting[\\"prompt_id\\"]) & set(validation[\\"prompt_id\\"]):\\n            raise ValueError(\\"Prompt leakage in Gemini fold assignments\\")\\n        if set(fitting[\\"model\\"]) != set(GEMINI_LABELS) or set(validation[\\"model\\"]) != set(GEMINI_LABELS):\\n            raise ValueError(\\"Every Gemini class must occur in fitting and validation for every fold\\")\\n    counts = {\\n        \\"train_rows\\": len(train), \\"gemini_rows\\": len(gemini),\\n        \\"selected_rows\\": int(gemini[\\"selected\\"].sum()),\\n        \\"baseline_selected_correct\\": int((gemini[\\"selected\\"] & gemini[\\"baseline_correct\\"]).sum()),\\n    }\\n    if expected_counts is not None:\\n        for key, expected in expected_counts.items():\\n            if counts.get(key) != expected:\\n                raise ValueError(f\\"Saved baseline cohort mismatch for {key}: expected {expected}, found {counts.get(key)}\\")\\n    protocol = {\\n        \\"counts\\": counts,\\n        \\"train_fingerprint\\": fingerprint_frame(train),\\n        \\"baseline_oof_fingerprint\\": _frame_digest(baseline, [\\"id\\", \\"prompt_id\\", \\"true_model\\", \\"fold\\", \\"predicted_model\\"]),\\n        \\"fold_metadata\\": fold_metadata,\\n        \\"labels\\": list(GEMINI_LABELS),\\n        \\"cohort_definition\\": \\"True model is Gemini AND saved CPU OOF prediction is Gemini; selection is for reporting only.\\",\\n        \\"training_population\\": \\"All true-Gemini rows in the fitting prompt folds, including CPU family false negatives.\\",\\n        \\"outer_validation_policy\\": \\"Fixed epochs; no checkpoint, epoch, hyperparameter or threshold selection on outer validation.\\",\\n    }\\n    return gemini, protocol\\n\\n\\ndef validate_model_snapshot(directory: str | Path) -> dict:\\n    directory = Path(directory)\\n    manifest = json.loads((directory / \\"snapshot_manifest.json\\").read_text())\\n    if (manifest.get(\\"schema_version\\") != 1 or manifest.get(\\"repo_id\\") != MODEL_ID\\n            or manifest.get(\\"revision\\") != MODEL_REVISION or manifest.get(\\"includes_weights\\") is not True):\\n        raise ValueError(\\"Expected a local snapshot of the pinned microsoft/deberta-v3-base revision\\")\\n    entries = manifest.get(\\"files\\", [])\\n    declared = set()\\n    if not entries:\\n        raise ValueError(\\"Snapshot manifest has no file inventory\\")\\n    for entry in entries:\\n        relative = Path(entry[\\"path\\"])\\n        if relative.is_absolute() or \\"..\\" in relative.parts or str(relative) in declared:\\n            raise ValueError(\\"Snapshot manifest contains an unsafe or repeated file path\\")\\n        declared.add(str(relative))\\n        target = directory / relative\\n        if target.is_symlink() or not target.is_file() or target.stat().st_size != entry[\\"size\\"]:\\n            raise ValueError(f\\"Missing or changed pinned snapshot file: {relative}\\")\\n    if not {\\"config.json\\", \\"spm.model\\", \\"pytorch_model.bin\\"}.issubset(declared):\\n        raise ValueError(\\"Pinned DeBERTa snapshot needs config.json, spm.model and pytorch_model.bin\\")\\n    architecture = json.loads((directory / \\"config.json\\").read_text())\\n    if (architecture.get(\\"model_type\\"), architecture.get(\\"hidden_size\\"),\\n            architecture.get(\\"num_hidden_layers\\")) != (\\"deberta-v2\\", 768, 12):\\n        raise ValueError(\\"Snapshot architecture does not match DeBERTa-v3-base\\")\\n    return manifest\\n\\n\\ndef validate_experiment_inputs(\\n    train_path: str | Path, baseline_oof_path: str | Path, fold_manifest_path: str | Path,\\n) -> dict:\\n    \\"\\"\\"CPU-only preflight before downloading a model or allocating GPU memory.\\"\\"\\"\\n    _, protocol = prepare_gemini_evaluation(\\n        read_train(train_path), pd.read_parquet(baseline_oof_path), fold_manifest_path,\\n        OFFICIAL_BASELINE_COUNTS,\\n    )\\n    return protocol\\n\\n\\ndef _seed_everything(seed: int, torch) -> None:\\n    random.seed(seed)\\n    np.random.seed(seed)\\n    torch.manual_seed(seed)\\n    if torch.cuda.is_available():\\n        torch.cuda.manual_seed_all(seed)\\n        torch.backends.cudnn.benchmark = False\\n        torch.backends.cuda.matmul.allow_tf32 = False\\n\\n\\ndef _initialize_cuda_memory_stats(torch, device) -> None:\\n    \\"\\"\\"Initialize this subprocess\'s caching allocator before resetting counters.\\"\\"\\"\\n    torch.cuda.init()\\n    torch.cuda.set_device(device)\\n    transient = torch.empty(1, device=device)\\n    del transient\\n    torch.cuda.empty_cache()\\n    torch.cuda.reset_peak_memory_stats(device)\\n\\n\\ndef _fit_predict_fold(\\n    fitting: pd.DataFrame, validation: pd.DataFrame, model_path: str | Path,\\n    config: GeminiConfig, fold: int,\\n) -> tuple[np.ndarray, dict]:\\n    \\"\\"\\"Full fine-tuning; a new pretrained backbone and random head per outer fold.\\"\\"\\"\\n    import torch\\n    from torch.utils.data import DataLoader\\n    from transformers import (\\n        AutoModelForSequenceClassification,\\n        AutoTokenizer,\\n        get_linear_schedule_with_warmup,\\n    )\\n\\n    config.validate()\\n    device = torch.device(config.device)\\n    cuda = device.type == \\"cuda\\"\\n    if cuda and not torch.cuda.is_available():\\n        raise ValueError(\\"This experiment requires a CUDA GPU; select a free Kaggle T4\\")\\n    if device.type not in {\\"cuda\\", \\"cpu\\"}:\\n        raise ValueError(\\"Use CUDA for the experiment or CPU for a tiny diagnostic\\")\\n    fold_seed = config.seed + fold\\n    _seed_everything(fold_seed, torch)\\n    started = time.perf_counter()\\n    if cuda:\\n        _initialize_cuda_memory_stats(torch, device)\\n    tokenizer = AutoTokenizer.from_pretrained(\\n        str(model_path), local_files_only=True, trust_remote_code=False, use_fast=False,\\n    )\\n    model = AutoModelForSequenceClassification.from_pretrained(\\n        str(model_path), local_files_only=True, trust_remote_code=False,\\n        use_safetensors=False, dtype=torch.float32, num_labels=len(GEMINI_LABELS),\\n        id2label=dict(enumerate(GEMINI_LABELS)),\\n        label2id={label: index for index, label in enumerate(GEMINI_LABELS)},\\n    ).to(device)\\n    if not all(parameter.requires_grad for parameter in model.parameters()):\\n        raise ValueError(\\"The experiment requires a fully trainable pretrained backbone and classifier\\")\\n\\n    def rows(frame):\\n        encoded = tokenizer(frame[\\"text\\"].tolist(), truncation=True, max_length=config.max_length)\\n        return [{**{name: values[index] for name, values in encoded.items()},\\n                 \\"labels\\": int(frame.iloc[index][\\"target\\"])} for index in range(len(frame))]\\n\\n    def collate(items):\\n        labels = torch.tensor([item[\\"labels\\"] for item in items], dtype=torch.long)\\n        batch = tokenizer.pad(\\n            [{key: value for key, value in item.items() if key != \\"labels\\"} for item in items],\\n            padding=True, return_tensors=\\"pt\\",\\n        )\\n        batch[\\"labels\\"] = labels\\n        return batch\\n\\n    fitting_rows, validation_rows = rows(fitting), rows(validation)\\n    generator = torch.Generator().manual_seed(fold_seed)\\n    train_loader = DataLoader(fitting_rows, batch_size=config.batch_size, shuffle=True,\\n                              collate_fn=collate, generator=generator, num_workers=0)\\n    valid_loader = DataLoader(validation_rows, batch_size=config.batch_size, shuffle=False,\\n                              collate_fn=collate, num_workers=0)\\n    decay, no_decay = [], []\\n    for name, parameter in model.named_parameters():\\n        (no_decay if parameter.ndim == 1 or name.endswith(\\".bias\\") else decay).append(parameter)\\n    optimizer = torch.optim.AdamW([\\n        {\\"params\\": decay, \\"weight_decay\\": config.weight_decay},\\n        {\\"params\\": no_decay, \\"weight_decay\\": 0.0},\\n    ], lr=config.learning_rate)\\n    updates_per_epoch = math.ceil(len(train_loader) / config.gradient_accumulation)\\n    total_updates = updates_per_epoch * config.epochs\\n    warmup_updates = math.ceil(config.warmup_ratio * total_updates)\\n    scheduler = get_linear_schedule_with_warmup(optimizer, warmup_updates, total_updates)\\n    scaler = torch.amp.GradScaler(\\"cuda\\", enabled=cuda, init_scale=config.amp_initial_scale)\\n    epoch_logs = []\\n    optimizer_updates = 0\\n    skipped_amp_updates = 0\\n    consecutive_amp_skips = 0\\n    try:\\n        for epoch in range(config.epochs):\\n            epoch_started = time.perf_counter()\\n            model.train()\\n            optimizer.zero_grad(set_to_none=True)\\n            total_loss = 0.0\\n            for step, batch in enumerate(train_loader):\\n                labels = batch.pop(\\"labels\\").to(device)\\n                inputs = {key: value.to(device) for key, value in batch.items()}\\n                group_start = (step // config.gradient_accumulation) * config.gradient_accumulation * config.batch_size\\n                group_rows = min(config.gradient_accumulation * config.batch_size, len(fitting_rows) - group_start)\\n                with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=cuda):\\n                    logits = model(**inputs).logits\\n                    summed_loss = torch.nn.functional.cross_entropy(logits.float(), labels, reduction=\\"sum\\")\\n                    loss = summed_loss / group_rows\\n                if not torch.isfinite(loss):\\n                    raise ValueError(f\\"Non-finite fitting loss in fold {fold}, epoch {epoch + 1}\\")\\n                total_loss += float(summed_loss.detach().cpu())\\n                scaler.scale(loss).backward()\\n                if (step + 1) % config.gradient_accumulation == 0 or step + 1 == len(train_loader):\\n                    scaler.unscale_(optimizer)\\n                    # Dynamic fp16 loss scaling may overflow scaled gradients even\\n                    # with a finite loss. GradScaler records that at unscale_,\\n                    # skips the unsafe update, and lowers its scale for retry.\\n                    torch.nn.utils.clip_grad_norm_(\\n                        model.parameters(), config.max_grad_norm, error_if_nonfinite=not cuda,\\n                    )\\n                    previous_scale = scaler.get_scale()\\n                    scaler.step(optimizer)\\n                    scaler.update()\\n                    if scaler.get_scale() < previous_scale:\\n                        skipped_amp_updates += 1\\n                        consecutive_amp_skips += 1\\n                        if consecutive_amp_skips >= config.max_consecutive_amp_skips:\\n                            raise ValueError(\\"Persistent non-finite CUDA gradients despite AMP scale backoff\\")\\n                    else:\\n                        scheduler.step()\\n                        optimizer_updates += 1\\n                        consecutive_amp_skips = 0\\n                    optimizer.zero_grad(set_to_none=True)\\n            epoch_logs.append({\\"epoch\\": epoch + 1, \\"mean_training_loss\\": total_loss / len(fitting_rows),\\n                               \\"elapsed_seconds\\": time.perf_counter() - epoch_started,\\n                               \\"learning_rate_after_epoch\\": scheduler.get_last_lr()[0]})\\n            print(json.dumps({\\"fold\\": fold, **epoch_logs[-1]}), flush=True)\\n        if optimizer_updates == 0:\\n            raise ValueError(\\"Fine-tuning completed without a successful optimizer update\\")\\n        # One validation pass after exactly the predeclared number of epochs.\\n        validation_started = time.perf_counter()\\n        model.eval()\\n        chunks = []\\n        with torch.inference_mode():\\n            for batch in valid_loader:\\n                batch.pop(\\"labels\\")\\n                inputs = {key: value.to(device) for key, value in batch.items()}\\n                with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=cuda):\\n                    logits = model(**inputs).logits\\n                chunks.append(torch.softmax(logits.float(), dim=-1).cpu().numpy())\\n        probabilities = np.concatenate(chunks)\\n        if not np.isfinite(probabilities).all() or not np.allclose(probabilities.sum(axis=1), 1.0, atol=1e-5):\\n            raise ValueError(\\"Non-finite or invalid DeBERTa validation probabilities\\")\\n        if cuda:\\n            torch.cuda.synchronize(device)\\n        log = {\\n            \\"fold\\": fold, \\"seed\\": fold_seed, \\"train_rows\\": len(fitting), \\"validation_rows\\": len(validation),\\n            \\"train_prompts\\": int(fitting[\\"prompt_id\\"].nunique()),\\n            \\"validation_prompts\\": int(validation[\\"prompt_id\\"].nunique()),\\n            \\"training_selected_rows\\": int(fitting[\\"selected\\"].sum()),\\n            \\"validation_selected_rows\\": int(validation[\\"selected\\"].sum()),\\n            \\"optimizer_updates\\": optimizer_updates, \\"skipped_amp_updates\\": skipped_amp_updates,\\n            \\"amp_initial_scale\\": config.amp_initial_scale if cuda else None,\\n            \\"amp_final_scale\\": scaler.get_scale() if cuda else None,\\n            \\"epochs\\": epoch_logs,\\n            \\"validation_elapsed_seconds\\": time.perf_counter() - validation_started,\\n            \\"elapsed_seconds\\": time.perf_counter() - started,\\n            \\"device\\": str(device), \\"gpu\\": torch.cuda.get_device_name(device) if cuda else None,\\n            \\"precision\\": \\"float32 parameters with CUDA float16 autocast/GradScaler\\" if cuda else \\"float32 CPU diagnostic\\",\\n            \\"peak_cuda_allocated_bytes\\": torch.cuda.max_memory_allocated(device) if cuda else None,\\n            \\"peak_cuda_reserved_bytes\\": torch.cuda.max_memory_reserved(device) if cuda else None,\\n        }\\n        return probabilities, log\\n    finally:\\n        del model, optimizer, scheduler, scaler\\n        gc.collect()\\n        if cuda:\\n            torch.cuda.empty_cache()\\n\\n\\ndef _cohort_metrics(frame: pd.DataFrame) -> dict:\\n    truth, prediction = frame[\\"model\\"], frame[\\"prediction\\"]\\n    baseline_correct = frame[\\"baseline_correct\\"].to_numpy(dtype=bool)\\n    specialist_correct = (truth == prediction).to_numpy(dtype=bool)\\n    return {\\n        \\"rows\\": len(frame), \\"baseline_correct\\": int(baseline_correct.sum()),\\n        \\"specialist_correct\\": int(specialist_correct.sum()),\\n        \\"baseline_accuracy\\": float(baseline_correct.mean()),\\n        \\"specialist_accuracy\\": float(accuracy_score(truth, prediction)),\\n        \\"specialist_balanced_accuracy\\": float(balanced_accuracy_score(truth, prediction)),\\n        \\"accuracy_delta\\": float(specialist_correct.mean() - baseline_correct.mean()),\\n        \\"paired_transitions\\": {\\n            \\"both_correct\\": int((baseline_correct & specialist_correct).sum()),\\n            \\"baseline_wrong_specialist_correct\\": int((~baseline_correct & specialist_correct).sum()),\\n            \\"baseline_correct_specialist_wrong\\": int((baseline_correct & ~specialist_correct).sum()),\\n            \\"both_wrong\\": int((~baseline_correct & ~specialist_correct).sum()),\\n        },\\n    }\\n\\n\\ndef _save_class_reports(frame: pd.DataFrame, directory: Path, suffix: str) -> None:\\n    precision, recall, f1, support = precision_recall_fscore_support(\\n        frame[\\"model\\"], frame[\\"prediction\\"], labels=list(GEMINI_LABELS), zero_division=0,\\n    )\\n    baseline_recall = frame.groupby(\\"model\\")[\\"baseline_correct\\"].mean().reindex(GEMINI_LABELS)\\n    pd.DataFrame({\\"model\\": GEMINI_LABELS, \\"precision\\": precision, \\"recall\\": recall,\\n                  \\"f1\\": f1, \\"support\\": support,\\n                  \\"baseline_recall\\": baseline_recall.to_numpy(),\\n                  \\"recall_delta\\": recall - baseline_recall.to_numpy()}).to_csv(\\n                      directory / f\\"class_metrics{suffix}.csv\\", index=False)\\n    pd.DataFrame(confusion_matrix(frame[\\"model\\"], frame[\\"prediction\\"], labels=list(GEMINI_LABELS)),\\n                 index=pd.Index(GEMINI_LABELS, name=\\"true_model\\"), columns=GEMINI_LABELS).to_csv(\\n                     directory / f\\"confusion_matrix{suffix}.csv\\")\\n\\n\\ndef run_experiment(\\n    train: pd.DataFrame, baseline_oof: pd.DataFrame, fold_manifest_path: str | Path,\\n    model_path: str | Path, output_dir: str | Path, config: GeminiConfig | None = None,\\n    expected_counts: dict[str, int] | None = OFFICIAL_BASELINE_COUNTS,\\n) -> dict:\\n    config = config or GeminiConfig()\\n    config.validate()\\n    gemini, protocol = prepare_gemini_evaluation(train, baseline_oof, fold_manifest_path, expected_counts)\\n    manifest = validate_model_snapshot(model_path)\\n    output = Path(output_dir)\\n    output.mkdir(parents=True, exist_ok=True)\\n    if (output / \\"metrics.json\\").exists() or (output / \\"oof.parquet\\").exists():\\n        raise ValueError(\\"Use a new output directory; an existing experiment must not be overwritten\\")\\n    versions = {}\\n    for package in (\\"torch\\", \\"transformers\\", \\"numpy\\", \\"pandas\\", \\"scikit-learn\\", \\"sentencepiece\\"):\\n        try:\\n            versions[package] = importlib.metadata.version(package)\\n        except importlib.metadata.PackageNotFoundError:\\n            versions[package] = None\\n    metadata = {\\"schema_version\\": 1, \\"kind\\": \\"gemini-four-class-full-finetuning-oof\\",\\n                \\"model_id\\": MODEL_ID, \\"model_revision\\": MODEL_REVISION,\\n                \\"configuration\\": asdict(config), \\"protocol\\": protocol, \\"versions\\": versions,\\n                \\"snapshot_manifest_sha256\\": hashlib.sha256(json.dumps(manifest, sort_keys=True).encode()).hexdigest(),\\n                \\"checkpoint_policy\\": \\"No full checkpoints saved; diagnostic OOF results only.\\",\\n                \\"seed_policy\\": \\"Fresh pretrained backbone/head each fold, seed + fold; bitwise CUDA determinism is not promised.\\"}\\n    _json(output / \\"metadata.json\\", metadata)\\n    full_manifest, fold_metadata = load_fold_manifest(train, fold_manifest_path)\\n    save_fold_manifest(full_manifest, fold_metadata, output)\\n    probabilities = np.full((len(gemini), len(GEMINI_LABELS)), np.nan)\\n    predicted_count = np.zeros(len(gemini), dtype=np.int64)\\n    training_logs = []\\n    started = time.perf_counter()\\n    for fold in range(5):\\n        fitting = gemini.loc[gemini[\\"fold\\"] != fold].copy()\\n        valid_mask = gemini[\\"fold\\"] == fold\\n        validation = gemini.loc[valid_mask].copy()\\n        # Fitting rows are deliberately not filtered by `selected`.\\n        if set(fitting[\\"prompt_id\\"]) & set(validation[\\"prompt_id\\"]):\\n            raise ValueError(\\"Prompt leakage detected before fine-tuning\\")\\n        values, log = _fit_predict_fold(fitting, validation, model_path, config, fold)\\n        if values.shape != (len(validation), len(GEMINI_LABELS)) or not np.isfinite(values).all():\\n            raise ValueError(\\"Fold prediction shape or finite-value validation failed\\")\\n        if (values < 0).any() or not np.allclose(values.sum(axis=1), 1.0, atol=1e-5):\\n            raise ValueError(\\"Fold probabilities must be nonnegative and sum to one\\")\\n        probabilities[valid_mask] = values\\n        predicted_count[valid_mask] += 1\\n        training_logs.append(log)\\n        _json(output / \\"training_log.json\\", training_logs)\\n    if not np.all(predicted_count == 1) or not np.isfinite(probabilities).all():\\n        raise ValueError(\\"Every true-Gemini row must receive exactly one OOF prediction\\")\\n    result = gemini[[\\"id\\", \\"prompt_id\\", \\"model\\", \\"fold\\", \\"baseline_model\\", \\"selected\\", \\"baseline_correct\\"]].copy()\\n    result[\\"prediction\\"] = np.asarray(GEMINI_LABELS)[probabilities.argmax(axis=1)]\\n    for index, label in enumerate(GEMINI_LABELS):\\n        result[f\\"probability::{label}\\"] = probabilities[:, index]\\n    selected = result.loc[result[\\"selected\\"]].copy()\\n    metrics = {\\"all_true_gemini\\": _cohort_metrics(result),\\n               \\"correctly_routed_gemini\\": _cohort_metrics(selected),\\n               \\"elapsed_seconds\\": time.perf_counter() - started,\\n               \\"interpretation\\": \\"Conditional exact-version attribution on saved CPU-routed true Gemini; not an end-to-end family router or hidden-test score.\\"}\\n    result.to_parquet(output / \\"oof.parquet\\", index=False)\\n    selected.to_parquet(output / \\"selected_oof.parquet\\", index=False)\\n    _save_class_reports(result, output, \\"\\")\\n    _save_class_reports(selected, output, \\"_selected\\")\\n    _json(output / \\"metrics.json\\", metrics)\\n    return metrics\\n\\n\\ndef main(argv: list[str] | None = None) -> int:\\n    parser = argparse.ArgumentParser(description=__doc__)\\n    parser.add_argument(\\"--train\\", type=Path, required=True)\\n    parser.add_argument(\\"--baseline-oof\\", type=Path, required=True)\\n    parser.add_argument(\\"--fold-manifest\\", type=Path, required=True)\\n    parser.add_argument(\\"--model-path\\", type=Path, required=True)\\n    parser.add_argument(\\"--output-dir\\", type=Path, required=True)\\n    parser.add_argument(\\"--epochs\\", type=int, default=3)\\n    parser.add_argument(\\"--learning-rate\\", type=float, default=2e-5)\\n    parser.add_argument(\\"--max-length\\", type=int, default=512)\\n    parser.add_argument(\\"--batch-size\\", type=int, default=8)\\n    parser.add_argument(\\"--gradient-accumulation\\", type=int, default=2)\\n    parser.add_argument(\\"--warmup-ratio\\", type=float, default=0.1)\\n    parser.add_argument(\\"--seed\\", type=int, default=2026)\\n    parser.add_argument(\\"--device\\", default=\\"cuda:0\\")\\n    args = parser.parse_args(argv)\\n    config = GeminiConfig(epochs=args.epochs, learning_rate=args.learning_rate,\\n                          max_length=args.max_length, batch_size=args.batch_size,\\n                          gradient_accumulation=args.gradient_accumulation,\\n                          warmup_ratio=args.warmup_ratio, seed=args.seed, device=args.device)\\n    metrics = run_experiment(read_train(args.train), pd.read_parquet(args.baseline_oof),\\n                             args.fold_manifest, args.model_path, args.output_dir, config)\\n    print(json.dumps(metrics, indent=2))\\n    return 0\\n\\n\\nif __name__ == \\"__main__\\":\\n    raise SystemExit(main())\\n", "ntl/modeling.py": "\\"\\"\\"Fold-local CPU training, aligned OOF scores and portable inference artifacts.\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport json\\nimport platform\\nfrom dataclasses import asdict, dataclass\\nfrom datetime import datetime, timezone\\nfrom pathlib import Path\\n\\nimport joblib\\nimport numpy as np\\nimport pandas as pd\\nimport sklearn\\nfrom sklearn.linear_model import LogisticRegression\\nfrom sklearn.metrics import (\\n    accuracy_score,\\n    balanced_accuracy_score,\\n    confusion_matrix,\\n    precision_recall_fscore_support,\\n)\\nfrom sklearn.pipeline import Pipeline\\n\\nfrom . import __version__\\nfrom .constants import DEFAULT_SEED, LABELS\\nfrom .data import (\\n    data_quality_report,\\n    fingerprint_frame,\\n    validate_test,\\n    validate_train,\\n    write_submission,\\n)\\nfrom .features import build_features\\nfrom .folds import build_fold_manifest, load_fold_manifest, save_fold_manifest\\n\\n\\n@dataclass(frozen=True)\\nclass BaselineConfig:\\n    seed: int = DEFAULT_SEED\\n    char_max_features: int = 50000\\n    word_max_features: int = 30000\\n    C: float = 4.0\\n    max_iter: int = 1000\\n\\n\\ndef build_pipeline(config: BaselineConfig | None = None) -> Pipeline:\\n    config = config or BaselineConfig()\\n    if config.C <= 0 or config.max_iter < 1 or min(config.char_max_features, config.word_max_features) < 1:\\n        raise ValueError(\\"Feature limits, C and max_iter must be positive\\")\\n    return Pipeline([\\n        (\\"features\\", build_features(config.char_max_features, config.word_max_features)),\\n        (\\"classifier\\", LogisticRegression(\\n            C=config.C, max_iter=config.max_iter, class_weight=\\"balanced\\", solver=\\"lbfgs\\",\\n            random_state=config.seed,\\n        )),\\n    ])\\n\\n\\ndef aligned_probabilities(pipeline: Pipeline, texts) -> np.ndarray:\\n    \\"\\"\\"Match estimator classes to LABELS explicitly; sklearn sorts labels itself.\\"\\"\\"\\n    classes = list(pipeline.named_steps[\\"classifier\\"].classes_)\\n    if len(classes) != len(LABELS) or set(classes) != set(LABELS):\\n        raise ValueError(\\"The fitted classifier does not contain all 18 competition labels\\")\\n    probabilities = pipeline.predict_proba(texts)\\n    if probabilities.shape != (len(texts), len(classes)):\\n        raise ValueError(\\"Unexpected classifier probability shape\\")\\n    aligned = probabilities[:, [classes.index(label) for label in LABELS]]\\n    if not np.isfinite(aligned).all() or not np.allclose(aligned.sum(axis=1), 1.0):\\n        raise ValueError(\\"Invalid prediction probabilities\\")\\n    return aligned\\n\\n\\ndef _write_json(path: Path, value: dict) -> None:\\n    path.write_text(json.dumps(value, indent=2, allow_nan=False) + \\"\\\\n\\")\\n\\n\\ndef _metadata(train: pd.DataFrame, config: BaselineConfig) -> dict:\\n    return {\\n        \\"schema_version\\": 1,\\n        \\"package_version\\": __version__,\\n        \\"created_at_utc\\": datetime.now(timezone.utc).isoformat(),\\n        \\"model_type\\": \\"char_word_stylometry_logistic_regression\\",\\n        \\"labels\\": list(LABELS),\\n        \\"train_fingerprint\\": fingerprint_frame(train),\\n        \\"n_rows\\": len(train),\\n        \\"n_prompts\\": int(train[\\"prompt_id\\"].nunique()),\\n        \\"configuration\\": asdict(config),\\n        \\"data_quality\\": data_quality_report(train),\\n        \\"versions\\": {\\"python\\": platform.python_version(), \\"scikit_learn\\": sklearn.__version__,\\n                     \\"numpy\\": np.__version__, \\"pandas\\": pd.__version__, \\"joblib\\": joblib.__version__},\\n        \\"text_processing\\": \\"Case and punctuation preserved; TF-IDF vocabularies and numeric scaling fit on training rows only.\\",\\n        \\"probability_note\\": \\"Class-weighted classifier probabilities are scores, not calibrated deployment posteriors.\\",\\n    }\\n\\n\\ndef run_cross_validation(\\n    train: pd.DataFrame, output_dir: str | Path, n_splits: int = 5,\\n    config: BaselineConfig | None = None, fold_manifest: str | Path | None = None,\\n) -> dict:\\n    train = validate_train(train).reset_index(drop=True)\\n    config = config or BaselineConfig()\\n    output = Path(output_dir)\\n    output.mkdir(parents=True, exist_ok=True)\\n    if fold_manifest is None:\\n        manifest, manifest_metadata = build_fold_manifest(train, n_splits=n_splits, seed=config.seed)\\n    else:\\n        manifest, manifest_metadata = load_fold_manifest(train, fold_manifest)\\n        if manifest_metadata[\\"n_splits\\"] != n_splits:\\n            raise ValueError(\\"Requested fold count differs from the supplied manifest\\")\\n    save_fold_manifest(manifest, manifest_metadata, output)\\n    folds = train[\\"id\\"].map(manifest.set_index(\\"id\\")[\\"fold\\"]).to_numpy()\\n    probabilities = np.full((len(train), len(LABELS)), np.nan)\\n    counts = np.zeros(len(train), dtype=np.int64)\\n    fold_metrics = []\\n    for fold in range(n_splits):\\n        fit_mask, valid_mask = folds != fold, folds == fold\\n        if set(train.loc[fit_mask, \\"prompt_id\\"]) & set(train.loc[valid_mask, \\"prompt_id\\"]):\\n            raise ValueError(\\"Prompt leakage detected before model fitting\\")\\n        # A new pipeline per fold ensures no vocabulary/scaler is fit on held-out rows.\\n        pipeline = build_pipeline(config)\\n        pipeline.fit(train.loc[fit_mask, \\"text\\"], train.loc[fit_mask, \\"model\\"])\\n        fold_probabilities = aligned_probabilities(pipeline, train.loc[valid_mask, \\"text\\"])\\n        probabilities[valid_mask] = fold_probabilities\\n        counts[valid_mask] += 1\\n        predictions = np.asarray(LABELS)[fold_probabilities.argmax(axis=1)]\\n        fold_metrics.append({\\n            \\"fold\\": fold,\\n            \\"n_train\\": int(fit_mask.sum()), \\"n_validation\\": int(valid_mask.sum()),\\n            \\"n_train_prompts\\": int(train.loc[fit_mask, \\"prompt_id\\"].nunique()),\\n            \\"n_validation_prompts\\": int(train.loc[valid_mask, \\"prompt_id\\"].nunique()),\\n            \\"balanced_accuracy\\": float(balanced_accuracy_score(train.loc[valid_mask, \\"model\\"], predictions)),\\n        })\\n    if not np.all(counts == 1) or not np.isfinite(probabilities).all():\\n        raise ValueError(\\"Every training row must receive exactly one finite OOF prediction\\")\\n    predictions = np.asarray(LABELS)[probabilities.argmax(axis=1)]\\n    oof = train[[\\"id\\", \\"prompt_id\\", \\"model\\"]].rename(columns={\\"model\\": \\"true_model\\"}).copy()\\n    oof[\\"fold\\"] = folds\\n    oof[\\"predicted_model\\"] = predictions\\n    for index, label in enumerate(LABELS):\\n        oof[f\\"probability::{label}\\"] = probabilities[:, index]\\n    oof.to_parquet(output / \\"oof.parquet\\", index=False)\\n    precision, recall, f1, support = precision_recall_fscore_support(\\n        train[\\"model\\"], predictions, labels=list(LABELS), zero_division=0,\\n    )\\n    pd.DataFrame({\\"model\\": LABELS, \\"precision\\": precision, \\"recall\\": recall,\\n                  \\"f1\\": f1, \\"support\\": support}).to_csv(output / \\"class_metrics.csv\\", index=False)\\n    pd.DataFrame(confusion_matrix(train[\\"model\\"], predictions, labels=list(LABELS)),\\n                 index=pd.Index(LABELS, name=\\"true_model\\"), columns=LABELS).to_csv(output / \\"confusion_matrix.csv\\")\\n    metrics = {\\n        \\"balanced_accuracy\\": float(balanced_accuracy_score(train[\\"model\\"], predictions)),\\n        \\"accuracy\\": float(accuracy_score(train[\\"model\\"], predictions)),\\n        \\"fold_balanced_accuracy_mean\\": float(np.mean([m[\\"balanced_accuracy\\"] for m in fold_metrics])),\\n        \\"fold_balanced_accuracy_std\\": float(np.std([m[\\"balanced_accuracy\\"] for m in fold_metrics])),\\n        \\"n_rows\\": len(train), \\"n_splits\\": n_splits, \\"folds\\": fold_metrics,\\n        \\"synthetic_data\\": bool(train[\\"id\\"].str.startswith(\\"synthetic-\\").all()),\\n    }\\n    _write_json(output / \\"metrics.json\\", metrics)\\n    metadata = _metadata(train, config)\\n    metadata[\\"validation\\"] = manifest_metadata\\n    _write_json(output / \\"metadata.json\\", metadata)\\n    return metrics\\n\\n\\ndef train_baseline(train: pd.DataFrame, output_dir: str | Path, config: BaselineConfig | None = None) -> Path:\\n    train = validate_train(train)\\n    config = config or BaselineConfig()\\n    output = Path(output_dir)\\n    output.mkdir(parents=True, exist_ok=True)\\n    pipeline = build_pipeline(config)\\n    pipeline.fit(train[\\"text\\"], train[\\"model\\"])\\n    # Fail early if an accidental training subset omitted or reordered labels.\\n    aligned_probabilities(pipeline, train[\\"text\\"].iloc[:1])\\n    metadata = _metadata(train, config)\\n    metadata[\\"estimator_classes\\"] = list(pipeline.named_steps[\\"classifier\\"].classes_)\\n    model_path = output / \\"model.joblib\\"\\n    joblib.dump({\\"pipeline\\": pipeline, \\"metadata\\": metadata}, model_path)\\n    _write_json(output / \\"metadata.json\\", metadata)\\n    return model_path\\n\\n\\ndef predict_baseline(model_path: str | Path, test: pd.DataFrame, output_path: str | Path) -> pd.DataFrame:\\n    test = validate_test(test)\\n    # joblib loads executable Python objects: only load artifacts you created/trust.\\n    artifact = joblib.load(model_path)\\n    metadata = artifact.get(\\"metadata\\", {})\\n    if metadata.get(\\"schema_version\\") != 1 or metadata.get(\\"labels\\") != list(LABELS):\\n        raise ValueError(\\"Unsupported artifact schema or label order\\")\\n    probabilities = aligned_probabilities(artifact[\\"pipeline\\"], test[\\"text\\"])\\n    predictions = np.asarray(LABELS)[probabilities.argmax(axis=1)].tolist()\\n    return write_submission(test[\\"id\\"], predictions, output_path)\\n", "ntl/neural.py": "\\"\\"\\"Offline frozen Qwen features and a prompt-grouped linear classification head.\\n\\nThis is a diagnostic baseline. It does not train LoRA adapters or claim a T4 benchmark.\\nRun ``python -m ntl.neural --help``. Neural dependencies are imported only when needed.\\n\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport argparse\\nimport gc\\nimport importlib.metadata\\nimport json\\nimport time\\nfrom dataclasses import asdict, dataclass\\nfrom pathlib import Path\\n\\nimport joblib\\nimport numpy as np\\nimport pandas as pd\\nfrom sklearn.linear_model import LogisticRegression\\nfrom sklearn.metrics import balanced_accuracy_score, confusion_matrix, recall_score\\nfrom sklearn.pipeline import make_pipeline\\nfrom sklearn.preprocessing import StandardScaler\\nfrom threadpoolctl import threadpool_limits\\n\\nfrom .constants import LABELS\\nfrom .data import fingerprint_frame, read_test, read_train, write_submission\\nfrom .folds import build_fold_manifest, load_fold_manifest, save_fold_manifest\\n\\nMODEL_ID = \\"Qwen/Qwen3-8B-Base\\"\\nMODEL_REVISION = \\"49e3418fbbbca6ecbdf9608b4d22e5a407081db4\\"\\n\\n\\n@dataclass(frozen=True)\\nclass FrozenConfig:\\n    retained_layers: int = 24\\n    pool_layers: tuple[int, ...] = (7, 22)\\n    quantization: str = \\"nf4\\"\\n    max_length: int = 512\\n    batch_size: int = 2\\n    C: float = 0.005\\n    seed: int = 2026\\n    device: str = \\"auto\\"\\n\\n    def validate(self) -> None:\\n        if not 1 <= self.retained_layers <= 36:\\n            raise ValueError(\\"retained_layers must be between 1 and 36\\")\\n        if not self.pool_layers or len(set(self.pool_layers)) != len(self.pool_layers):\\n            raise ValueError(\\"pool_layers must be nonempty and unique\\")\\n        if any(layer < 1 or layer > self.retained_layers for layer in self.pool_layers):\\n            raise ValueError(\\"Each pooled layer must be a retained decoder block (1-based)\\")\\n        if self.quantization not in {\\"none\\", \\"nf4\\"}:\\n            raise ValueError(\\"quantization must be none or nf4\\")\\n        if self.max_length < 16 or self.batch_size < 1 or self.C <= 0:\\n            raise ValueError(\\"Invalid token limit, batch size, or regularization\\")\\n\\n\\ndef validate_snapshot(directory: str | Path) -> dict:\\n    \\"\\"\\"Reject mutable/wrong model inputs before loading any serialized weights.\\"\\"\\"\\n    directory = Path(directory)\\n    manifest_path = directory / \\"snapshot_manifest.json\\"\\n    if not manifest_path.is_file():\\n        raise ValueError(\\"Missing snapshot_manifest.json; run scripts/prepare_model.py first\\")\\n    manifest = json.loads(manifest_path.read_text())\\n    if (\\n        manifest.get(\\"schema_version\\") != 1\\n        or manifest.get(\\"repo_id\\") != MODEL_ID\\n        or manifest.get(\\"revision\\") != MODEL_REVISION\\n        or manifest.get(\\"includes_weights\\") is not True\\n    ):\\n        raise ValueError(\\"Snapshot must contain the pinned Qwen3-8B-Base weights\\")\\n    entries = manifest.get(\\"files\\", [])\\n    if not entries:\\n        raise ValueError(\\"Snapshot has no recorded files\\")\\n    for item in entries:\\n        relative = Path(item[\\"path\\"])\\n        if relative.is_absolute() or \\"..\\" in relative.parts:\\n            raise ValueError(\\"Invalid path in snapshot manifest\\")\\n        target = directory / relative\\n        if not target.is_file() or target.stat().st_size != item[\\"size\\"]:\\n            raise ValueError(f\\"Snapshot file missing or size changed: {relative}\\")\\n    if not any(str(item[\\"path\\"]).endswith(\\".safetensors\\") for item in entries):\\n        raise ValueError(\\"Expected safetensors model weights\\")\\n    architecture = json.loads((directory / \\"config.json\\").read_text())\\n    if (\\n        architecture.get(\\"model_type\\") != \\"qwen3\\"\\n        or architecture.get(\\"num_hidden_layers\\") != 36\\n        or architecture.get(\\"hidden_size\\") != 4096\\n    ):\\n        raise ValueError(\\"Snapshot architecture is not the selected full Qwen3-8B-Base\\")\\n    return manifest\\n\\n\\ndef masked_layer_mean(hidden_states, attention_mask, layers: tuple[int, ...]):\\n    \\"\\"\\"Concatenate float32 means; padding must contribute neither mass nor count.\\"\\"\\"\\n    import torch\\n\\n    mask = attention_mask.unsqueeze(-1).to(dtype=torch.float32)\\n    counts = mask.sum(dim=1)\\n    if torch.any(counts == 0):\\n        raise ValueError(\\"Cannot pool an empty token sequence\\")\\n    pooled = [(hidden_states[layer].float() * mask).sum(dim=1) / counts for layer in layers]\\n    return torch.cat(pooled, dim=-1)\\n\\n\\ndef _versions() -> dict:\\n    versions = {}\\n    for package in (\\"torch\\", \\"transformers\\", \\"bitsandbytes\\", \\"numpy\\", \\"scikit-learn\\"):\\n        try:\\n            versions[package] = importlib.metadata.version(package)\\n        except importlib.metadata.PackageNotFoundError:\\n            versions[package] = None\\n    return versions\\n\\n\\ndef extract_features(\\n    texts: list[str], model_path: str | Path, config: FrozenConfig\\n) -> tuple[np.ndarray, dict]:\\n    \\"\\"\\"Read only a local pinned snapshot, with no network fallback or remote code.\\"\\"\\"\\n    config.validate()\\n    manifest = validate_snapshot(model_path)\\n    try:\\n        import torch\\n        from transformers import AutoConfig, AutoModel, AutoTokenizer, BitsAndBytesConfig\\n    except ImportError as exc:\\n        raise RuntimeError(\\"Install the neural extra; NF4 also needs the Linux gpu extra\\") from exc\\n    device = config.device\\n    if device == \\"auto\\":\\n        device = \\"cuda:0\\" if torch.cuda.is_available() else \\"cpu\\"\\n    if device.startswith(\\"cuda\\") and not torch.cuda.is_available():\\n        raise ValueError(\\"CUDA requested but unavailable\\")\\n    if config.quantization == \\"nf4\\" and not device.startswith(\\"cuda\\"):\\n        raise ValueError(\\"NF4 requires a CUDA GPU; use --quantization none for CPU diagnostics\\")\\n    if not texts or any(not isinstance(text, str) or not text.strip() for text in texts):\\n        raise ValueError(\\"Expected nonempty passage strings\\")\\n    torch.manual_seed(config.seed)\\n    if device.startswith(\\"cuda\\"):\\n        torch.cuda.manual_seed_all(config.seed)\\n        torch.cuda.reset_peak_memory_stats(device)\\n    started = time.perf_counter()\\n    tokenizer = AutoTokenizer.from_pretrained(\\n        str(model_path), local_files_only=True, trust_remote_code=False, padding_side=\\"right\\"\\n    )\\n    if tokenizer.pad_token_id is None:\\n        if tokenizer.eos_token_id is None:\\n            raise ValueError(\\"Tokenizer has neither pad nor EOS token\\")\\n        tokenizer.pad_token = tokenizer.eos_token\\n    architecture = AutoConfig.from_pretrained(\\n        str(model_path), local_files_only=True, trust_remote_code=False\\n    )\\n    architecture.num_hidden_layers = config.retained_layers\\n    architecture.use_cache = False\\n    dtype = torch.float16 if device.startswith(\\"cuda\\") else torch.float32\\n    load_args = {\\n        \\"config\\": architecture,\\n        \\"local_files_only\\": True,\\n        \\"trust_remote_code\\": False,\\n        \\"dtype\\": dtype,\\n        \\"attn_implementation\\": \\"sdpa\\",\\n        \\"use_safetensors\\": True,\\n        \\"low_cpu_mem_usage\\": True,\\n        \\"device_map\\": {\\"\\": device},\\n    }\\n    if config.quantization == \\"nf4\\":\\n        load_args[\\"quantization_config\\"] = BitsAndBytesConfig(\\n            load_in_4bit=True,\\n            bnb_4bit_quant_type=\\"nf4\\",\\n            bnb_4bit_use_double_quant=True,\\n            bnb_4bit_compute_dtype=torch.float16,\\n        )\\n    # AutoModel loads Qwen3Model, avoiding the unused language-model vocabulary head.\\n    # num_hidden_layers controls instantiation, so discarded upper blocks never reside on GPU.\\n    model = AutoModel.from_pretrained(str(model_path), **load_args).eval()\\n    chunks = []\\n    token_lengths: list[int] = []\\n    try:\\n        with torch.inference_mode():\\n            for start in range(0, len(texts), config.batch_size):\\n                batch_text = texts[start : start + config.batch_size]\\n                lengths = tokenizer(batch_text, truncation=False, return_length=True)[\\"length\\"]\\n                token_lengths.extend(lengths)\\n                encoded = tokenizer(\\n                    batch_text,\\n                    padding=True,\\n                    truncation=True,\\n                    max_length=config.max_length,\\n                    return_tensors=\\"pt\\",\\n                ).to(device)\\n                output = model(**encoded, output_hidden_states=True, use_cache=False)\\n                pooled = masked_layer_mean(\\n                    output.hidden_states, encoded[\\"attention_mask\\"], config.pool_layers\\n                )\\n                chunks.append(pooled.cpu().numpy())\\n                del output, pooled, encoded\\n        if device.startswith(\\"cuda\\"):\\n            torch.cuda.synchronize(device)\\n        profile = {\\n            \\"elapsed_seconds\\": time.perf_counter() - started,\\n            \\"device\\": device,\\n            \\"device_name\\": torch.cuda.get_device_name(device) if device.startswith(\\"cuda\\") else \\"CPU\\",\\n            \\"compute_dtype\\": str(dtype),\\n            \\"peak_cuda_allocated_bytes\\": (\\n                torch.cuda.max_memory_allocated(device) if device.startswith(\\"cuda\\") else None\\n            ),\\n            \\"rows\\": len(texts),\\n            \\"truncated_rows\\": sum(length > config.max_length for length in token_lengths),\\n            \\"token_length_quantiles\\": {\\n                str(q): float(np.quantile(token_lengths, q)) for q in (0.5, 0.9, 0.99, 1.0)\\n            },\\n            \\"model_id\\": manifest[\\"repo_id\\"],\\n            \\"model_revision\\": manifest[\\"revision\\"],\\n            \\"versions\\": _versions(),\\n        }\\n        features = np.concatenate(chunks).astype(np.float32, copy=False)\\n        if not np.isfinite(features).all():\\n            raise ValueError(\\"Backbone produced non-finite features; inspect precision and inputs\\")\\n        return features, profile\\n    finally:\\n        del model\\n        gc.collect()\\n        if device.startswith(\\"cuda\\"):\\n            torch.cuda.empty_cache()\\n\\n\\ndef _head(config: FrozenConfig):\\n    return make_pipeline(\\n        StandardScaler(),\\n        LogisticRegression(C=config.C, max_iter=2000, random_state=config.seed),\\n    )\\n\\n\\ndef _json(path: Path, value: dict) -> None:\\n    path.write_text(json.dumps(value, indent=2) + \\"\\\\n\\")\\n\\n\\ndef _labels(train: pd.DataFrame) -> np.ndarray:\\n    return np.array([LABELS.index(label) for label in train[\\"model\\"]], dtype=np.int64)\\n\\n\\ndef _probabilities(head, features: np.ndarray) -> np.ndarray:\\n    if not np.array_equal(head.classes_, np.arange(len(LABELS))):\\n        raise ValueError(\\"Classifier does not contain the complete canonical label order\\")\\n    probabilities = head.predict_proba(features)\\n    if not np.isfinite(probabilities).all():\\n        raise ValueError(\\"Classifier produced non-finite probabilities\\")\\n    return probabilities\\n\\n\\ndef run_cv(\\n    train: pd.DataFrame,\\n    model_path: str | Path,\\n    output_dir: str | Path,\\n    config: FrozenConfig,\\n    n_splits: int = 5,\\n    fold_manifest: str | Path | None = None,\\n) -> dict:\\n    manifest, fold_metadata = (\\n        load_fold_manifest(train, fold_manifest)\\n        if fold_manifest is not None\\n        else build_fold_manifest(train, n_splits=n_splits, seed=config.seed)\\n    )\\n    features, profile = extract_features(train[\\"text\\"].tolist(), model_path, config)\\n    targets = _labels(train)\\n    assignments = train[\\"id\\"].map(manifest.set_index(\\"id\\")[\\"fold\\"]).to_numpy()\\n    oof = np.full((len(train), len(LABELS)), np.nan, dtype=np.float64)\\n    scores = []\\n    for fold in range(fold_metadata[\\"n_splits\\"]):\\n        fit, valid = assignments != fold, assignments == fold\\n        head = _head(config)\\n        with threadpool_limits(limits=2):\\n            head.fit(features[fit], targets[fit])\\n            oof[valid] = _probabilities(head, features[valid])\\n        scores.append(float(balanced_accuracy_score(targets[valid], oof[valid].argmax(axis=1))))\\n    if not np.isfinite(oof).all():\\n        raise ValueError(\\"OOF predictions are incomplete or non-finite\\")\\n    predictions = oof.argmax(axis=1)\\n    metrics = {\\n        \\"balanced_accuracy\\": float(balanced_accuracy_score(targets, predictions)),\\n        \\"fold_scores\\": scores,\\n        \\"labels\\": list(LABELS),\\n        \\"config\\": asdict(config),\\n        \\"train_fingerprint\\": fingerprint_frame(train),\\n        \\"model_id\\": MODEL_ID,\\n        \\"model_revision\\": MODEL_REVISION,\\n        \\"validation\\": \\"prompt-grouped; frozen unsupervised feature extraction, fold-fitted heads\\",\\n    }\\n    directory = Path(output_dir)\\n    directory.mkdir(parents=True, exist_ok=True)\\n    save_fold_manifest(manifest, fold_metadata, directory)\\n    frame = train[[\\"id\\", \\"prompt_id\\", \\"model\\"]].copy()\\n    frame[\\"fold\\"] = assignments\\n    frame[\\"prediction\\"] = [LABELS[index] for index in predictions]\\n    for index, label in enumerate(LABELS):\\n        frame[f\\"probability__{label}\\"] = oof[:, index]\\n    frame.to_parquet(directory / \\"oof.parquet\\", index=False)\\n    pd.DataFrame({\\n        \\"model\\": LABELS,\\n        \\"recall\\": recall_score(targets, predictions, labels=np.arange(len(LABELS)), average=None),\\n    }).to_csv(directory / \\"class_metrics.csv\\", index=False)\\n    pd.DataFrame(\\n        confusion_matrix(targets, predictions, labels=np.arange(len(LABELS))),\\n        index=pd.Index(LABELS, name=\\"true_model\\"), columns=LABELS,\\n    ).to_csv(directory / \\"confusion_matrix.csv\\")\\n    _json(directory / \\"metrics.json\\", metrics)\\n    _json(directory / \\"feature_profile.json\\", profile)\\n    return metrics\\n\\n\\ndef train_head(\\n    train: pd.DataFrame, model_path: str | Path, output_dir: str | Path, config: FrozenConfig\\n) -> dict:\\n    features, profile = extract_features(train[\\"text\\"].tolist(), model_path, config)\\n    head = _head(config)\\n    with threadpool_limits(limits=2):\\n        head.fit(features, _labels(train))\\n    metadata = {\\n        \\"schema_version\\": 1,\\n        \\"kind\\": \\"frozen-qwen-linear-head\\",\\n        \\"labels\\": list(LABELS),\\n        \\"model_id\\": MODEL_ID,\\n        \\"model_revision\\": MODEL_REVISION,\\n        \\"snapshot_path\\": str(Path(model_path).resolve()),\\n        \\"config\\": asdict(config),\\n        \\"train_fingerprint\\": fingerprint_frame(train),\\n        \\"n_train\\": len(train),\\n        \\"versions\\": _versions(),\\n    }\\n    directory = Path(output_dir)\\n    directory.mkdir(parents=True, exist_ok=True)\\n    joblib.dump({\\"head\\": head, \\"metadata\\": metadata}, directory / \\"head.joblib\\")\\n    _json(directory / \\"metadata.json\\", metadata)\\n    _json(directory / \\"feature_profile.json\\", profile)\\n    return metadata\\n\\n\\ndef predict_head(\\n    model_dir: str | Path,\\n    test: pd.DataFrame,\\n    output: str | Path,\\n    model_path: str | Path | None = None,\\n) -> pd.DataFrame:\\n    # joblib is trusted project output; it must never load an untrusted third-party pickle.\\n    artifact = joblib.load(Path(model_dir) / \\"head.joblib\\")\\n    metadata = artifact[\\"metadata\\"]\\n    if (\\n        metadata.get(\\"kind\\") != \\"frozen-qwen-linear-head\\"\\n        or metadata.get(\\"labels\\") != list(LABELS)\\n        or metadata.get(\\"model_id\\") != MODEL_ID\\n        or metadata.get(\\"model_revision\\") != MODEL_REVISION\\n    ):\\n        raise ValueError(\\"Incompatible neural artifact metadata or label order\\")\\n    config_values = dict(metadata[\\"config\\"])\\n    config_values[\\"pool_layers\\"] = tuple(config_values[\\"pool_layers\\"])\\n    config = FrozenConfig(**config_values)\\n    features, profile = extract_features(\\n        test[\\"text\\"].tolist(), model_path or metadata[\\"snapshot_path\\"], config\\n    )\\n    with threadpool_limits(limits=2):\\n        probabilities = _probabilities(artifact[\\"head\\"], features)\\n    result = write_submission(test[\\"id\\"], [LABELS[i] for i in probabilities.argmax(axis=1)], output)\\n    output_path = Path(output)\\n    _json(output_path.with_name(f\\"{output_path.stem}_profile.json\\"), profile)\\n    return result\\n\\n\\ndef main(argv: list[str] | None = None) -> int:\\n    parser = argparse.ArgumentParser(description=__doc__)\\n    commands = parser.add_subparsers(dest=\\"command\\", required=True)\\n    for name in (\\"cv\\", \\"train\\"):\\n        sub = commands.add_parser(name)\\n        sub.add_argument(\\"--train\\", required=True, type=Path)\\n        sub.add_argument(\\"--model-path\\", required=True, type=Path)\\n        sub.add_argument(\\"--output-dir\\", required=True, type=Path)\\n        sub.add_argument(\\"--quantization\\", choices=(\\"none\\", \\"nf4\\"), default=\\"nf4\\")\\n        sub.add_argument(\\"--batch-size\\", type=int, default=2)\\n        sub.add_argument(\\"--max-length\\", type=int, default=512)\\n        sub.add_argument(\\"--retained-layers\\", type=int, default=24)\\n        sub.add_argument(\\"--pool-layers\\", type=int, nargs=\\"+\\", default=[7, 22])\\n        sub.add_argument(\\"--C\\", type=float, default=0.005)\\n        sub.add_argument(\\"--seed\\", type=int, default=2026)\\n        sub.add_argument(\\"--device\\", default=\\"auto\\")\\n        if name == \\"cv\\":\\n            sub.add_argument(\\"--folds\\", type=int, default=5)\\n            sub.add_argument(\\"--fold-manifest\\", type=Path)\\n    predict = commands.add_parser(\\"predict\\")\\n    predict.add_argument(\\"--model-dir\\", required=True, type=Path)\\n    predict.add_argument(\\"--model-path\\", type=Path)\\n    predict.add_argument(\\"--test\\", required=True, type=Path)\\n    predict.add_argument(\\"--output\\", required=True, type=Path)\\n    args = parser.parse_args(argv)\\n    if args.command == \\"predict\\":\\n        result = predict_head(args.model_dir, read_test(args.test), args.output, args.model_path)\\n        print(f\\"Wrote {len(result)} predictions to {args.output}\\")\\n        return 0\\n    config = FrozenConfig(\\n        retained_layers=args.retained_layers, pool_layers=tuple(args.pool_layers),\\n        quantization=args.quantization, max_length=args.max_length, batch_size=args.batch_size,\\n        C=args.C, seed=args.seed, device=args.device,\\n    )\\n    config.validate()\\n    train = read_train(args.train)\\n    if args.command == \\"cv\\":\\n        result = run_cv(\\n            train, args.model_path, args.output_dir, config, args.folds, args.fold_manifest\\n        )\\n        print(f\\"Prompt-grouped balanced accuracy: {result[\'balanced_accuracy\']:.5f}\\")\\n    else:\\n        train_head(train, args.model_path, args.output_dir, config)\\n        print(f\\"Saved frozen feature classifier to {args.output_dir}\\")\\n    return 0\\n\\n\\nif __name__ == \\"__main__\\":\\n    raise SystemExit(main())\\n", "ntl/smoke.py": "\\"\\"\\"Clearly synthetic fixtures for exercising plumbing, never competition scoring.\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport json\\nfrom pathlib import Path\\n\\nimport pandas as pd\\n\\nfrom .constants import LABELS\\nfrom .modeling import BaselineConfig, predict_baseline, run_cross_validation, train_baseline\\n\\n\\ndef synthetic_frames(n_prompts: int = 6) -> tuple[pd.DataFrame, pd.DataFrame]:\\n    \\"\\"\\"Artificial class markers make these data unsuitable for modeling claims.\\"\\"\\"\\n    rows = []\\n    for prompt in range(n_prompts):\\n        for model_index, label in enumerate(LABELS):\\n            rows.append({\\n                \\"id\\": f\\"synthetic-{prompt:03d}-{model_index:02d}\\",\\n                \\"prompt_id\\": f\\"synthetic-prompt-{prompt:03d}\\",\\n                \\"text\\": (f\\"SYNTHETIC FIXTURE ONLY. ClassMarker{model_index:02d} \\"\\n                         f\\"ClassMarker{model_index:02d}. Prompt theme {prompt}; \\"\\n                         \\"Case, punctuation\\u2014\\u2018quotes\\u2019\\u2014and line breaks\\\\nare preserved. \\"\\n                         \\"This manufactured sample is not an authentic model response.\\"),\\n                \\"model\\": label,\\n            })\\n    train = pd.DataFrame(rows)\\n    test = pd.DataFrame({\\n        \\"id\\": [\\"synthetic-test-000\\", \\"synthetic-test-001\\", \\"synthetic-test-002\\"],\\n        \\"text\\": [rows[index][\\"text\\"] for index in (0, 7, 17)],\\n    })\\n    return train, test\\n\\n\\ndef run_smoke(output_dir: str | Path) -> dict:\\n    output = Path(output_dir)\\n    output.mkdir(parents=True, exist_ok=True)\\n    train, test = synthetic_frames()\\n    train.to_csv(output / \\"synthetic_train.csv\\", index=False)\\n    test.to_csv(output / \\"synthetic_test.csv\\", index=False)\\n    config = BaselineConfig(char_max_features=3000, word_max_features=1000, max_iter=300)\\n    cv = run_cross_validation(train, output / \\"cv\\", n_splits=5, config=config)\\n    model_path = train_baseline(train, output / \\"model\\", config=config)\\n    submission = predict_baseline(model_path, test, output / \\"submission.parquet\\")\\n    summary = {\\"synthetic_data\\": True, \\"purpose\\": \\"Pipeline plumbing only; scores have no competition meaning.\\",\\n               \\"training_rows\\": len(train), \\"test_rows\\": len(test), \\"submission_rows\\": len(submission),\\n               \\"cv_completed\\": True, \\"model_saved\\": str(model_path), \\"cv\\": cv}\\n    (output / \\"smoke_summary.json\\").write_text(json.dumps(summary, indent=2) + \\"\\\\n\\")\\n    return summary\\n"}'
SOURCE_SHA256 = 'a61b1bb9affdc2ae72469ae10d3ec8adc7052a808c3b5ecd2259874ca8404b7a'
assert hashlib.sha256(SOURCE_JSON.encode()).hexdigest() == SOURCE_SHA256
SOURCE_ROOT = Path("/kaggle/working/ntl_source")
for relative, content in json.loads(SOURCE_JSON).items():
    target = SOURCE_ROOT / relative
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(content, encoding="utf-8")
sys.path.insert(0, str(SOURCE_ROOT))
importlib.invalidate_caches()
print("Embedded ntl source SHA-256:", SOURCE_SHA256)


In [ ]:
REQUIRED_PACKAGES = ['numpy==2.3.5', 'pandas==2.2.3', 'scipy==1.16.2', 'scikit-learn==1.7.2', 'pyarrow==21.0.0', 'joblib==1.5.2', 'threadpoolctl==3.6.0', 'packaging==26.3']
WHEEL_PROFILE = 'cpu'
WHEEL_REQUIREMENTS_SHA256 = '25d5b83424e7d0ef29846c4facc63db88576cf85a6d62e0ed9650e1c33f3a80a'
# Attach the pinned wheel bundle from kaggle_prepare_dependencies.ipynb.
# Set an explicit folder if more than one bundle is attached.
WHEEL_DIR = None  # directory containing wheel_manifest.json and requirements.lock
if WHEEL_DIR is None:
    wheel_candidates = []
    for wheel_manifest in sorted(Path("/kaggle/input").rglob("wheel_manifest.json")):
        try:
            candidate_manifest = json.loads(wheel_manifest.read_text())
        except (OSError, ValueError):
            continue
        if candidate_manifest.get("profile") == WHEEL_PROFILE:
            wheel_candidates.append(wheel_manifest)
    if len(wheel_candidates) == 1:
        WHEEL_DIR = str(wheel_candidates[0].parent)
    elif len(wheel_candidates) > 1:
        raise RuntimeError("Several wheel bundles are attached; set WHEEL_DIR explicitly")
if WHEEL_DIR is None:
    raise RuntimeError(
        f"Attach the prepared {WHEEL_PROFILE} wheel bundle from kaggle_prepare_dependencies.ipynb"
    )
import platform
import subprocess
import venv
bundle_manifest = json.loads((Path(WHEEL_DIR) / "wheel_manifest.json").read_text())
if bundle_manifest.get("profile") != WHEEL_PROFILE:
    raise RuntimeError(f"Attach the {WHEEL_PROFILE} wheel bundle for this notebook")
if (bundle_manifest.get("python") != list(sys.version_info[:2])
        or bundle_manifest.get("machine") != platform.machine()
        or bundle_manifest.get("system") != platform.system()):
    raise RuntimeError("Prepare the wheel bundle on the same Kaggle Python/Linux image as this run")
wheel_requirements_path = Path(WHEEL_DIR) / "requirements.lock"
if not wheel_requirements_path.is_file():
    raise RuntimeError("Wheel bundle is missing requirements.lock")
if hashlib.sha256(wheel_requirements_path.read_bytes()).hexdigest() != WHEEL_REQUIREMENTS_SHA256:
    raise RuntimeError("Wheel requirements differ from this notebook; attach its matching prepared bundle")

# Train and infer in a fresh package environment, independent of packages that
# Kaggle's notebook kernel may already have imported (including audio/vision).
RUNTIME_DIR = Path("/kaggle/working/ntl_env") / (WHEEL_PROFILE + "-" + WHEEL_REQUIREMENTS_SHA256[:12])
# Kaggle can disable ensurepip; the notebook's existing pip can install into
# a pip-less virtual environment through its supported --python option.
venv.EnvBuilder(with_pip=False, system_site_packages=False).create(RUNTIME_DIR)
RUNTIME_PYTHON = str(RUNTIME_DIR / "bin" / "python")
RUNTIME_ENV = dict(os.environ)
RUNTIME_ENV.pop("PYTHONHOME", None)
RUNTIME_ENV["PYTHONPATH"] = str(SOURCE_ROOT)
RUNTIME_ENV["PYTHONNOUSERSITE"] = "1"
RUNTIME_ENV["PIP_DISABLE_PIP_VERSION_CHECK"] = "1"
subprocess.run(
    [sys.executable, "-m", "pip", "--python", RUNTIME_PYTHON,
     "install", "--no-index", "--find-links",
     str(WHEEL_DIR), "--require-hashes", "-r", str(wheel_requirements_path)],
    check=True, env=RUNTIME_ENV,
)
version_probe = """
from importlib.metadata import PackageNotFoundError, version
from packaging.requirements import Requirement
import json
import sys
for raw_requirement in json.loads(sys.argv[1]):
    requirement = Requirement(raw_requirement)
    if requirement.marker and not requirement.marker.evaluate():
        continue
    package = requirement.name
    try:
        installed_version = version(package)
    except PackageNotFoundError as exc:
        raise RuntimeError(
            f"Missing {package} in the isolated runtime; recreate the matching wheel bundle."
        ) from exc
    if installed_version not in requirement.specifier:
        raise RuntimeError(
            f"Expected {requirement}; installed {package}=={installed_version}. "
            "Recreate the matching wheel bundle before rerunning."
        )
    print(package, installed_version)
"""
subprocess.run([RUNTIME_PYTHON, "-c", version_probe, json.dumps(REQUIRED_PACKAGES)],
               check=True, env=RUNTIME_ENV)
def run_runtime(module, arguments):
    subprocess.run([RUNTIME_PYTHON, "-m", module, *arguments],
                   check=True, env=RUNTIME_ENV)
print("Isolated runtime:", RUNTIME_PYTHON)


In [ ]:
import csv

# Usually leave this unset. Never discover train.csv recursively: attached
# external datasets may also contain a file with that name.
COMPETITION_DIR = None
official_locations = [
    Path("/kaggle/input/competitions/name-that-llm"),
    Path("/kaggle/input/name-that-llm"),
]
if COMPETITION_DIR is not None:
    official_locations = [Path(COMPETITION_DIR)]
available = [p for p in official_locations
             if (p / "train.csv").is_file() and (p / "test.csv").is_file()]
if len(available) != 1:
    raise RuntimeError(
        "Attach the official Name That LLM competition data. If both official "
        "paths exist, set COMPETITION_DIR explicitly. Found: " + str(available)
    )
DATA_DIR = available[0]
TRAIN = str(DATA_DIR / "train.csv")
TEST = str(DATA_DIR / "test.csv")
SUBMISSION = "/kaggle/working/submission.parquet"
# A failed fresh run must not leave an earlier prediction file available.
Path(SUBMISSION).unlink(missing_ok=True)
def read_input_csv(path, name):
    with open(path, encoding="utf-8-sig", newline="") as handle:
        reader = csv.DictReader(handle)
        if not {"id", "text"}.issubset(reader.fieldnames or []):
            raise ValueError(f"{name} requires id and text columns")
        rows = list(reader)
    if not rows or any(not row["id"] or not row["text"] for row in rows):
        raise ValueError(f"{name} is empty or contains null IDs/text")
    ids = [row["id"] for row in rows]
    if len(set(ids)) != len(ids):
        raise ValueError(f"{name} has duplicate IDs")
    return rows
train_rows = read_input_csv(TRAIN, "train")
test_rows = read_input_csv(TEST, "test")
VALID_LABELS = {
    "gpt-5.5", "gpt-5.6-luna", "gpt-5.6-sol", "gpt-5.6-terra", "gpt-6-astra",
    "gemini-3.1-pro-preview", "gemini-3.6-flash", "gemini-3.7-flash", "gemini-3.8-flash",
    "claude-fable-5", "claude-opus-4-6", "claude-opus-4-8", "claude-opus-5", "claude-sonnet-5",
    "qwen3.8-max", "glm-5.3", "glm-5.3-flash", "kimi-k3",
}
if {row.get("model") for row in train_rows} != VALID_LABELS:
    raise ValueError("Official training labels do not match the expected 18 labels")
print(f"Official train: {len(train_rows):,} rows; current test: {len(test_rows):,} rows")
print("The visible test is a smoke test; scoring replaces it with the hidden test.")


In [ ]:
def run_cpu(arguments):
    run_runtime("ntl", arguments)
RUN_GROUPED_CV = False
if RUN_GROUPED_CV:
    run_cpu(["baseline-cv", "--train", TRAIN,
             "--output-dir", "/kaggle/working/cpu-cv"])


In [ ]:
CPU_MODEL_DIR = "/kaggle/working/cpu-model"
run_cpu(["baseline-train", "--train", TRAIN, "--output-dir", CPU_MODEL_DIR])
run_cpu(["predict", "--model", CPU_MODEL_DIR + "/model.joblib",
         "--test", TEST, "--output", SUBMISSION])


In [ ]:
# Validate the file the competition will score, including arbitrary test size.
validation_code = """
import csv
import json
import sys
import pandas as pd
SUBMISSION, TEST, labels_json = sys.argv[1:]
VALID_LABELS = set(json.loads(labels_json))
with open(TEST, encoding="utf-8-sig", newline="") as handle:
    expected_ids = [row["id"] for row in csv.DictReader(handle)]
submission = pd.read_parquet(SUBMISSION)
if list(submission.columns) != ["id", "model"]:
    raise ValueError("submission.parquet must contain exactly id, model in that order")
if len(submission) != len(expected_ids):
    raise ValueError("Submission row count differs from the current test")
if submission.isna().any().any() or submission["id"].duplicated().any():
    raise ValueError("Submission contains null values or duplicate IDs")
if submission["id"].astype("string").tolist() != expected_ids:
    raise ValueError("Submission must preserve the official test IDs and row order")
if not submission["model"].isin(VALID_LABELS).all():
    raise ValueError("Submission contains a model outside the 18 allowed labels")
print(f"Validated {len(submission):,} predictions: {SUBMISSION}")
print(submission["model"].value_counts().to_string())
print(submission.head().to_string(index=False))
"""
subprocess.run([RUNTIME_PYTHON, "-c", validation_code, SUBMISSION, TEST,
                json.dumps(sorted(VALID_LABELS))], check=True, env=RUNTIME_ENV)


Submit the **saved notebook version's `submission.parquet`** through
Kaggle. The hidden rerun executes this same code against every hidden
test passage. The artifacts under `cpu-model/` are fitted on all
training rows and do not represent a held-out validation result.
